### QS-PM / GQRS-PM Implementácia - User Friendly  v1.36

In [ ]:
# Colab setup: clone repo and install dependencies
import os
if not os.path.exists('/content/BakalarkaUf'):
    os.system('git clone https://github.com/Gura-hub-hub/BakalarkaUf.git /content/BakalarkaUf')
os.chdir('/content/BakalarkaUf')
os.system('pip install pmdarima -q')


### [1] Načítanie knižníc a funkcii  

In [ ]:
class _W:
    def __init__(self, v): self.value = v

w_dataset   = _W('sk');   w_model     = _W('qspm')
w_mhat      = _W('4');    w_p_win     = _W('8')
w_r         = _W('1');    w_rb        = _W('0')
w_center    = _W('none'); w_fc_g      = _W('auto_arima')
w_fc_gb     = _W('auto_arima');      w_transform = _W('log')
w_center_fc = _W('linear_anchored')
w_gp        = _W('0');    w_gd        = _W('1');  w_gq = _W('3')
w_gbp       = _W('1');    w_gbd       = _W('1');  w_gbq = _W('0')
import json as _j, IPython as _ipy
try:
    from google.colab import _message as _m
    _nb = _m.blocking_request("get_ipynb", request="", timeout_sec=5)
    if isinstance(_nb, str): _nb = _j.loads(_nb)
    if 'ipynb' in _nb: _nb = _nb['ipynb']
except Exception:
    import os as _os, glob as _gl
    _nb_name = "Bakalarka_uf.ipynb"
    _nb_path = next((p for p in [_nb_name, f"/content/{_nb_name}", f"/content/BakalarkaUf/{_nb_name}"] + _gl.glob(f"**/{_nb_name}", recursive=True) if _os.path.exists(p)), _nb_name)
    _nb = _j.load(open(_nb_path, encoding="utf-8"))
_sep = next(i for i, c in enumerate(_nb['cells'])
            if c['cell_type'] == 'markdown' and 'Defin' in ''.join(c['source']))
for _c in _nb['cells'][_sep + 1:]:
    if _c['cell_type'] == 'code':
        _src = chr(10).join('' if l.strip().startswith('%') or l.strip().startswith('!')
                          else l for l in ''.join(_c['source']).splitlines())
        _ipy.get_ipython().run_cell(_src)
del _j, _nb, _sep, _c, _src, _ipy
_uf_load_data()

### [2] Nastavenie parametrov

In [ ]:
_uf_show_panel()

### [3] Execution modelu so zvolenými parametrami

In [ ]:
_uf_show_run_button()

### [4] Printout výsledkov

In [ ]:
_uf_show_results()

### [5] Printout analýzy

In [ ]:
_uf_show_diagnostics()

---
## –– Definície ––

*Bunky nižšie obsahujú implementáciu modelov. Pri bežnom používaní ich nie je potrebné meniť.*

In [65]:
import pickle as _pk, os as _os, pandas as _pd_uf
from IPython.display import display, HTML as _HTML

_DATASETS = {
    'sk': 'data/Slovensko_exporty_2004_2025.csv',
    'cz10c': 'data/testsets/cz_exports_10c_2010-2024/exports.csv',
    'cz30c': 'data/testsets/cz_exports_30c_2010-2024/exports.csv',
    'ap': 'data/AirPassengers_values.csv',
}
_SARIMA_PKLS = {
    'sk': 'data/lists/sarima_prediction_[sk_ex_8w].pkl',
    'cz10c': 'data/lists/sarima_10c_p8.pkl',
    'cz30c': 'data/lists/sarima_30c_p8.pkl',
    'ap': 'data/lists/sarima_prediction_[ap_test].pkl',
}
_START_YEARS = {'sk': 2004, 'cz10c': 2010, 'cz30c': 2010, 'ap': 1949}

_PANEL_HTML = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap');
*,*::before,*::after{box-sizing:border-box}
.uf-panel{
  font-family:'Inter',-apple-system,BlinkMacSystemFont,sans-serif;
  font-size:13px;
  background:#0c0e16;
  border:1px solid #1e2438;
  padding:18px 24px;
  width:780px;
  color:#ffffff;
}
.uf-panel *{font-family:inherit;color:inherit}
.uf-sec{margin-bottom:16px}
.uf-sec-title{
  font-size:10px;
  font-weight:600;
  letter-spacing:1.5px;
  color:#5b78c4;
  text-transform:uppercase;
  margin-bottom:8px;
}
.uf-row{display:flex;flex-wrap:wrap;align-items:center;gap:6px;margin-top:0}
.uf-row+.uf-row{margin-top:6px}
.uf-field{display:inline-flex;align-items:center;gap:5px}
.uf-label{white-space:nowrap;font-size:13px;color:#ffffff;padding-right:2px}
.uf-tip{
  position:relative;
  display:inline-flex;align-items:center;justify-content:center;
  width:11px;height:11px;
  border-radius:50%;
  background:#131925;
  border:1px solid #1e2a3a;
  color:#3d5882;
  font-size:7px;font-weight:600;
  cursor:default;
  margin-left:3px;
  line-height:1;
  flex-shrink:0;
  vertical-align:middle;
  opacity:0.65;
  transition:opacity 0.15s;
}
.uf-tip:hover{opacity:1}
.uf-tip-box{
  display:none;
  position:fixed;
  background:#0d1120;
  color:#ffffff;
  border:1px solid #2a3452;
  border-radius:6px;
  padding:7px 10px;
  font-size:11px;
  line-height:1.55;
  width:200px;
  z-index:99999;
  text-align:left;
  font-weight:normal;
  white-space:normal;
  pointer-events:none;
  box-shadow:0 4px 20px rgba(0,0,0,0.7);
}
.uf-sep{
  width:1px;min-width:1px;
  background:linear-gradient(to bottom,transparent 0%,#1e2438 30%,#1e2438 70%,transparent 100%);
  height:24px;margin:0 4px;flex-shrink:0;
}
.uf-input{
  background:#1e2e40;
  color:#ffffff;
  border:1px solid #2e3550;
  border-radius:5px;
  padding:3px 7px;
  height:26px;
  font-family:'Inter',sans-serif;
  font-size:13px;
  outline:none;
  width:60px;
}
.uf-input:focus{border-color:#5b78c4;box-shadow:0 0 0 2px rgba(91,120,196,0.22)}
.uf-dd{position:relative;display:inline-block}
.uf-dd-trg{
  display:flex;align-items:center;justify-content:space-between;
  background:#1e2e40;color:#ffffff;
  border:1px solid #2e3550;border-radius:5px;
  padding:3px 8px;cursor:pointer;
  font-size:13px;font-family:'Inter',sans-serif;
  height:26px;user-select:none;white-space:nowrap;width:100%;
}
.uf-dd-trg:hover{border-color:#5b78c4}
.uf-dd-arr{margin-left:8px;color:#5b78c4;font-size:10px;flex-shrink:0}
.uf-dd-menu{
  display:none;position:absolute;
  top:calc(100% + 2px);left:0;
  background:#0d1120;border:1px solid #2a3452;border-radius:7px;
  z-index:9998;min-width:100%;
  box-shadow:0 4px 16px rgba(0,0,0,0.55);padding:3px 0;
}
.uf-dd-opt{
  display:flex;align-items:center;justify-content:space-between;
  padding:6px 10px;cursor:pointer;
  color:#ffffff;font-size:13px;font-family:'Inter',sans-serif;
  white-space:nowrap;gap:8px;
}
.uf-dd-opt:hover{background:#151c2e}

.uf-pdq{display:none;align-items:center;gap:6px}
.uf-pdq.visible{display:flex}
</style>
<script>
(function(){
  window.ufSet=function(name,val){
    google.colab.kernel.invokeFunction('uf_set_param',[name,val===null?'__none__':String(val)],{});
  };
  window.ufInput=function(el){ufSet(el.dataset.name,el.value)};
  window.ufDdToggle=function(trg){
    var menu=trg.closest('.uf-dd').querySelector('.uf-dd-menu');
    var open=menu.style.display==='block';
    document.querySelectorAll('.uf-dd-menu').forEach(function(m){m.style.display='none'});
    if(!open)menu.style.display='block';
  };
  window.ufDdSel=function(opt){
    var dd=opt.closest('.uf-dd');
    var name=dd.dataset.name;
    var raw=opt.dataset.val;
    var pyval=raw==='__none__'?null:raw;
    dd.querySelector('.uf-dd-val').textContent=opt.querySelector('.uf-dd-lbl').textContent;
    dd.querySelector('.uf-dd-menu').style.display='none';
    dd.dataset.current=raw;
    ufSet(name,pyval);
    if(name==='w_model')  ufModelDefaults(raw);
    if(name==='w_center')  ufToggleCenterFc(raw);
    if(name==='w_fc_g')   ufTogglePdq(raw,'pdq-g');
    if(name==='w_fc_gb')  ufTogglePdq(raw,'pdq-gb');
  };
  window.ufTogglePdq=function(val,gid){
    var g=document.getElementById(gid);if(!g)return;
    if(val==='__none__')g.classList.add('visible');
    else g.classList.remove('visible');
  };
  window.ufToggleCenterFc=function(val){
    var g=document.getElementById('center-fc-group');if(!g)return;
    g.style.display=val==='none'?'none':'flex';
  };
  window.ufModelDefaults=function(model){
    var D={
      qspm:  {w_mhat:'4',w_p_win:'8',w_r:'1',w_rb:'0',w_center:'none',  w_fc_g:'auto_arima',      w_fc_gb:'auto_arima',      w_center_fc:'linear_anchored', w_transform:'log'},
      gqrspm:{w_mhat:'4',w_p_win:'8',w_r:'1',w_rb:'0',w_center:'row',   w_fc_g:'linear_anchored', w_fc_gb:'linear_anchored', w_center_fc:'linear_anchored', w_transform:'log'}
    };
    var d=D[model];if(!d)return;
    Object.keys(d).forEach(function(k){
      ufSet(k,d[k]);
      var el=document.querySelector('[data-name="'+k+'"]');if(!el)return;
      if(el.classList.contains('uf-dd')){
        var dv=d[k]||'__none__';
        var opt=el.querySelector('[data-val="'+dv+'"]');
        if(opt)el.querySelector('.uf-dd-val').textContent=opt.querySelector('.uf-dd-lbl').textContent;
        el.dataset.current=dv;
        if(k==='w_center') ufToggleCenterFc(dv);
        if(k==='w_fc_g')  ufTogglePdq(dv,'pdq-g');
        if(k==='w_fc_gb') ufTogglePdq(dv,'pdq-gb');
      }else if(el.tagName==='INPUT'){
        el.value=d[k];
      }
    });
  };
  document.addEventListener('click',function(e){
    if(!e.target.closest||!e.target.closest('.uf-dd'))
      document.querySelectorAll('.uf-dd-menu').forEach(function(m){m.style.display='none'});
  },true);
  window.ufApplyAll=function(){
    var p={};
    document.querySelectorAll('input[data-name]').forEach(function(el){p[el.dataset.name]=el.value;});
    document.querySelectorAll('.uf-dd[data-name]').forEach(function(dd){
      p[dd.dataset.name]=dd.dataset.current||dd.querySelector('.uf-dd-opt').dataset.val;
    });
    var btn=document.getElementById('uf-apply-btn');
    btn.innerHTML='&#10003;&nbsp;Aplikované';
    setTimeout(function(){btn.innerHTML='&#10003;&nbsp;Použiť parametre';},1200);
    if(typeof google!=='undefined'&&google.colab&&google.colab.kernel){
      google.colab.kernel.invokeFunction('uf_apply_all',[JSON.stringify(p)],{});
      return;
    }
  };
  setTimeout(function(){
    document.querySelectorAll('.uf-dd[data-name]').forEach(function(dd){
      var first=dd.querySelector('.uf-dd-opt');if(first)dd.dataset.current=first.dataset.val;
    });
    document.querySelectorAll('.uf-tip').forEach(function(t){
      var b=t.querySelector('.uf-tip-box');if(!b)return;
      t.addEventListener('mouseenter',function(){
        var r=t.getBoundingClientRect();
        b.style.top=(r.top+r.height/2)+'px';
        b.style.left=(r.right+10)+'px';
        b.style.transform='translateY(-50%)';
        b.style.display='block';
      });
      t.addEventListener('mouseleave',function(){b.style.display='none'});
    });
  },300);
})();
</script>
<div class="uf-panel">

  <div class="uf-sec">
    <div class="uf-sec-title">Dataset &amp; Model</div>
    <div class="uf-row">

      <div class="uf-field">
        <span class="uf-label">Dataset<span class="uf-tip">i<span class="uf-tip-box">zdrojový časový rad</span></span></span>
        <div class="uf-dd" data-name="w_dataset" style="width:160px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">Slovensko (SK)</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="sk"  onclick="ufDdSel(this)"><span class="uf-dd-lbl">Slovensko (SK)</span><span class="uf-tip">i<span class="uf-tip-box">slovenský export 2004–2025</span></span></div>
            <div class="uf-dd-opt" data-val="cz10c" onclick="ufDdSel(this)"><span class="uf-dd-lbl">CZ-10C</span><span class="uf-tip">i<span class="uf-tip-box">Czech export – 10 krajín, 2010–2024</span></span></div>
            <div class="uf-dd-opt" data-val="cz30c" onclick="ufDdSel(this)"><span class="uf-dd-lbl">CZ-30C</span><span class="uf-tip">i<span class="uf-tip-box">Czech export – 30 krajín, 2010–2024</span></span></div>
          </div>
        </div>
      </div>

      <div class="uf-sep"></div>

      <div class="uf-field">
        <span class="uf-label">Model</span>
        <div class="uf-dd" data-name="w_model" style="width:160px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">QS-PM</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="qspm"   onclick="ufDdSel(this)"><span class="uf-dd-lbl">QS-PM</span></div>
            <div class="uf-dd-opt" data-val="gqrspm" onclick="ufDdSel(this)"><span class="uf-dd-lbl">GQRS-PM</span></div>
          </div>
        </div>
      </div>

    </div>
  </div>

  <div class="uf-sec">
    <div class="uf-sec-title">Okno &amp; Komponenty</div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">m&#x0302;<span class="uf-tip">i<span class="uf-tip-box">veľkosť okna v rokoch</span></span></span>
        <input class="uf-input" type="text" data-name="w_mhat"  value="4" oninput="ufInput(this)">
      </div>
      <div class="uf-sep"></div>
      <div class="uf-field">
        <span class="uf-label">p<span class="uf-tip">i<span class="uf-tip-box">počet testovacích okien</span></span></span>
        <input class="uf-input" type="text" data-name="w_p_win" value="8" oninput="ufInput(this)">
      </div>
    </div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">r<span class="uf-tip">i<span class="uf-tip-box">počet regulárnych SVD komponentov</span></span></span>
        <input class="uf-input" type="text" data-name="w_r"     value="1" oninput="ufInput(this)">
      </div>
      <div class="uf-sep"></div>
      <div class="uf-field">
        <span class="uf-label">r<sub>b</sub><span class="uf-tip">i<span class="uf-tip-box">počet neregulárnych SVD komponentov</span></span></span>
        <input class="uf-input" type="text" data-name="w_rb"    value="0" oninput="ufInput(this)">
      </div>
    </div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">Centering<span class="uf-tip">i<span class="uf-tip-box">centrovanie matice pred SVD</span></span></span>
        <div class="uf-dd" data-name="w_center" style="width:130px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">none</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="none"   onclick="ufDdSel(this)"><span class="uf-dd-lbl">none</span><span class="uf-tip">i<span class="uf-tip-box">bez centrovania</span></span></div>
            <div class="uf-dd-opt" data-val="column" onclick="ufDdSel(this)"><span class="uf-dd-lbl">column</span><span class="uf-tip">i<span class="uf-tip-box">odčíta priemer stĺpcov</span></span></div>
            <div class="uf-dd-opt" data-val="row"    onclick="ufDdSel(this)"><span class="uf-dd-lbl">row</span><span class="uf-tip">i<span class="uf-tip-box">odčíta priemer riadkov</span></span></div>
            <div class="uf-dd-opt" data-val="double" onclick="ufDdSel(this)"><span class="uf-dd-lbl">double</span><span class="uf-tip">i<span class="uf-tip-box">Kombinuje row aj column centrovanie.</span></span></div>
          </div>
        </div>
      </div>
      <div id="center-fc-group" style="display:none;align-items:center;gap:6px">
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">Forecaster</span>
          <div class="uf-dd" data-name="w_center_fc" style="width:200px">
            <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">AUTOARIMA</span><span class="uf-dd-arr">&#9660;</span></div>
            <div class="uf-dd-menu">
              <div class="uf-dd-opt" data-val="auto_arima"      onclick="ufDdSel(this)"><span class="uf-dd-lbl">AUTOARIMA</span><span class="uf-tip">i<span class="uf-tip-box">pmdarima auto_arima: automatický výber p,d,q.</span></span></div>
              <div class="uf-dd-opt" data-val="linear_anchored" onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear Anchored</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia ukotvená na posledný bod série</span></span></div>
              <div class="uf-dd-opt" data-val="linear"          onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia</span></span></div>
              <div class="uf-dd-opt" data-val="__none__"        onclick="ufDdSel(this)"><span class="uf-dd-lbl">ARIMA (manuál)</span><span class="uf-tip">i<span class="uf-tip-box">ARIMA s manuálnym p,d,q. Zadaj hodnoty v poliach vpravo.</span></span></div>
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>

  <div class="uf-sec">
    <div class="uf-sec-title">G-séria</div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">g<span class="uf-tip">i<span class="uf-tip-box" style="white-space:nowrap;width:auto">forecaster regulárnej g-série</span></span></span>
        <div class="uf-dd" data-name="w_fc_g" style="width:200px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">AUTOARIMA</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="auto_arima"      onclick="ufDdSel(this)"><span class="uf-dd-lbl">AUTOARIMA</span><span class="uf-tip">i<span class="uf-tip-box">pmdarima auto_arima: automatický výber p,d,q.</span></span></div>
            <div class="uf-dd-opt" data-val="linear_anchored" onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear Anchored</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia ukotvená na posledný bod série</span></span></div>
            <div class="uf-dd-opt" data-val="linear"          onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia</span></span></div>
            <div class="uf-dd-opt" data-val="__none__"        onclick="ufDdSel(this)"><span class="uf-dd-lbl">ARIMA (manuál)</span><span class="uf-tip">i<span class="uf-tip-box">ARIMA s manuálnym p,d,q. Zadaj hodnoty v poliach vpravo.</span></span></div>
          </div>
        </div>
      </div>
      <div class="uf-pdq" id="pdq-g">
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">p</span>
          <input class="uf-input" type="text" data-name="w_gp" value="0" oninput="ufInput(this)" style="width:48px">
        </div>
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">d</span>
          <input class="uf-input" type="text" data-name="w_gd" value="1" oninput="ufInput(this)" style="width:48px">
        </div>
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">q</span>
          <input class="uf-input" type="text" data-name="w_gq" value="3" oninput="ufInput(this)" style="width:48px">
        </div>
      </div>
    </div>
  </div>

  <div class="uf-sec">
    <div class="uf-sec-title">G<sub>b</sub>-séria</div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">g<sub>b</sub><span class="uf-tip">i<span class="uf-tip-box" style="white-space:nowrap;width:auto">forecaster neregulárnej g_b-série (po transformácii)</span></span></span>
        <div class="uf-dd" data-name="w_fc_gb" style="width:200px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">AUTOARIMA</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="auto_arima"      onclick="ufDdSel(this)"><span class="uf-dd-lbl">AUTOARIMA</span><span class="uf-tip">i<span class="uf-tip-box">pmdarima auto_arima: automatický výber p,d,q.</span></span></div>
            <div class="uf-dd-opt" data-val="linear_anchored" onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear Anchored</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia ukotvená na posledný bod série</span></span></div>
            <div class="uf-dd-opt" data-val="linear"          onclick="ufDdSel(this)"><span class="uf-dd-lbl">Linear</span><span class="uf-tip">i<span class="uf-tip-box">lineárna regresia</span></span></div>
            <div class="uf-dd-opt" data-val="__none__"        onclick="ufDdSel(this)"><span class="uf-dd-lbl">ARIMA (manuál)</span><span class="uf-tip">i<span class="uf-tip-box">ARIMA s manuálnym p,d,q. Zadaj hodnoty v poliach vpravo.</span></span></div>
          </div>
        </div>
      </div>
      <div class="uf-pdq" id="pdq-gb">
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">p</span>
          <input class="uf-input" type="text" data-name="w_gbp" value="1" oninput="ufInput(this)" style="width:48px">
        </div>
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">d</span>
          <input class="uf-input" type="text" data-name="w_gbd" value="1" oninput="ufInput(this)" style="width:48px">
        </div>
        <div class="uf-sep"></div>
        <div class="uf-field">
          <span class="uf-label">q</span>
          <input class="uf-input" type="text" data-name="w_gbq" value="0" oninput="ufInput(this)" style="width:48px">
        </div>
      </div>
    </div>
  </div>

  <div class="uf-sec" style="margin-bottom:0">
    <div class="uf-sec-title">Transformácia</div>
    <div class="uf-row">
      <div class="uf-field">
        <span class="uf-label">Transform<span class="uf-tip">i<span class="uf-tip-box">Transformácia aplikovaná na A_N pred SVD neregulárnej časti.</span></span></span>
        <div class="uf-dd" data-name="w_transform" style="width:100px">
          <div class="uf-dd-trg" onclick="ufDdToggle(this)"><span class="uf-dd-val">log</span><span class="uf-dd-arr">&#9660;</span></div>
          <div class="uf-dd-menu">
            <div class="uf-dd-opt" data-val="log"  onclick="ufDdSel(this)"><span class="uf-dd-lbl">log</span><span class="uf-tip">i<span class="uf-tip-box">logaritmus A_N (ln)</span></span></div>
            <div class="uf-dd-opt" data-val="sqrt" onclick="ufDdSel(this)"><span class="uf-dd-lbl">sqrt</span><span class="uf-tip">i<span class="uf-tip-box">druhá odmocnina A_N</span></span></div>
          </div>
        </div>
      </div>
    </div>
  </div>

  <div style="display:flex;justify-content:flex-end;margin-top:16px;padding-top:12px;border-top:1px solid #1e2438;position:sticky;bottom:0;background:#0c0e16;z-index:10">
    <button id="uf-apply-btn" onclick="ufApplyAll()" style="background:#0d1e30;color:#7aaae8;border:1px solid #2a4a6a;border-radius:5px;font-family:Inter,sans-serif;font-size:12px;font-weight:600;padding:5px 18px;cursor:pointer;letter-spacing:0.3px;transition:background 0.15s,border-color 0.15s">&#10003; Použiť parametre</button>
  </div>

</div>
"""


def _uf_load_data():
    global DATASET, export_vector, export_dict, n, m, start_year, sarima_prediction
    DATASET       = w_dataset.value
    if DATASET in ("cz10c", "cz30c"):
        _df = _pd_uf.read_csv(_DATASETS[DATASET])
        if _df.columns[0].lower() in ("month","date","time","year"): _df = _df.iloc[:, 1:]
        export_dict   = {col: _df[col].values.astype(float).tolist() for col in _df.columns}
        export_vector = list(next(iter(export_dict.values())))
    else:
        export_dict   = None
        export_vector = _pd_uf.read_csv(_DATASETS[DATASET], header=None).squeeze().tolist()
    n             = 12
    m             = len(export_vector) // n
    start_year    = _START_YEARS.get(DATASET, 2004)
    _pkl = _SARIMA_PKLS.get(DATASET)
    sarima_prediction = None
    if _pkl and _os.path.exists(_pkl):
        with open(_pkl, 'rb') as _f:
            sarima_prediction = _pk.load(_f)
def _uf_show_panel():
    try:
        import google.colab.output as _co_p, json as _jp
        def _set_param(name, val):
            _g = globals()
            if name in _g:
                _g[name].value = None if val == '__none__' else val
        def _apply_all(json_str):
            _g = globals()
            for _k, _v in _jp.loads(json_str).items():
                if _k in _g:
                    _g[_k].value = None if _v == '__none__' else _v
            return 'ok'
        _co_p.register_callback('uf_set_param', _set_param)
        _co_p.register_callback('uf_apply_all', _apply_all)
    except Exception:
        pass
    display(_HTML(_PANEL_HTML))



def _uf_show_run_button():
    import json as _jsc, threading as _thr, time as _t2
    from IPython.display import display as _dsp, HTML as _HTML
    import google.colab.output as _co

    _uid   = format(int(_t2.time() * 1000) % 999983, 'x')
    _state = {'pct': 0.0, 'eta': '', 'running': False, 'done': False, 'error': ''}
    _stop  = _thr.Event()

    def _cb_get():
        return _jsc.dumps(_state)

    def _cb_stop():
        _stop.set()
        return 'ok'

    def _cb_start(params_json='{}'):
        if _state['running']:
            return 'running'
        _p2 = _jsc.loads(params_json)
        def _gv(k, fallback):
            v = _p2.get(k, fallback)
            return None if v == '__none__' else v
        model      = _gv('w_model', w_model.value)
        m_hat      = int(_gv('w_mhat', w_mhat.value))
        p          = int(_gv('w_p_win', w_p_win.value))
        r          = int(_gv('w_r', w_r.value))
        r_b        = int(_gv('w_rb', w_rb.value))
        g_orders   = [(int(_gv('w_gp', w_gp.value)), int(_gv('w_gd', w_gd.value)), int(_gv('w_gq', w_gq.value)))]
        g_b_orders = [(int(_gv('w_gbp', w_gbp.value)), int(_gv('w_gbd', w_gbd.value)), int(_gv('w_gbq', w_gbq.value)))]
        g_fc       = _gv('w_fc_g', w_fc_g.value)
        gb_fc      = _gv('w_fc_gb', w_fc_gb.value)
        transform  = _gv('w_transform', w_transform.value)
        centering  = _gv('w_center', w_center.value)
        center_fc  = _gv('w_center_fc', w_center_fc.value)
        w_model.value = model
        clf = {i: 'r_n' for i in range(r)}
        for _i in range(r, m_hat):
            clf[_i] = 'r_b'
        clf['r'] = r;  clf['r_b'] = r_b
        dataset   = _gv('w_dataset', w_dataset.value)
        try:
            _ds_cur = DATASET
        except NameError:
            _ds_cur = None
        if _ds_cur != dataset:
            w_dataset.value = dataset
            _uf_load_data()
        _stop.clear()
        _state.update({'pct': 0.0, 'eta': 'Spustam...', 'running': True, 'done': False, 'error': ''})

        def _worker():
            import time as _tm, warnings
            global qspm_prediction, gqrspm_prediction
            total = p;  times = []
            try:
                _cur_ctry = None; _cur_win = None
                if dataset in ("cz10c", "cz30c"):
                    # MV worker
                    if export_dict is None:
                        raise RuntimeError("[MV] export_dict je None - spusti _uf_load_data() pre " + dataset)
                    _ctrs = list(export_dict.keys())
                    total = p * len(_ctrs); _done = 0
                    if model == "qspm":
                        clear_g_store()
                        qspm_prediction = {}
                        for _ctry in _ctrs:
                            _cur_ctry = _ctry
                            _ser = export_dict[_ctry]
                            _m_c = len(_ser) // n
                            qspm_prediction[_ctry] = []
                            for _i in range(p):
                                if _stop.is_set(): break
                                _cur_win = _i
                                _t0 = _tm.time()
                                _cut = _ser[:n * (_m_c - (_i + 1))]
                                _pred = one_period_prediction_classified(
                                    m_hat, _m_c - (_i + 1), clf, n, _cut,
                                    g_orders=g_orders, g_b_orders=g_b_orders,
                                    g_forecaster=g_fc, g_b_forecaster=gb_fc,
                                    transform=transform, power_p=0.5, r_b=r_b, center=centering,
                                    column_mean_order=(1, 1, 0), row_mean_order=(1, 1, 0),
                                    full_series=_ser,
                                )
                                qspm_prediction[_ctry].append(_pred)
                                _done += 1; times.append(_tm.time() - _t0)
                                _pct = _done / total * 100
                                _avg = sum(times) / len(times)
                                _rem = (total - _done) * _avg
                                _eta = f"{int(_rem//60)}m {int(_rem%60)}s" if _rem >= 60 else f"{int(_rem)+1}s"
                                _state.update({'pct': _pct, 'eta': _eta})
                            if _stop.is_set(): break
                    elif model == "gqrspm":
                        clear_g_store()
                        gqrspm_prediction = {}
                        for _ctry in _ctrs:
                            _cur_ctry = _ctry
                            _ser = export_dict[_ctry]
                            _m_c = len(_ser) // n
                            gqrspm_prediction[_ctry] = []
                            for _i in range(p):
                                if _stop.is_set(): break
                                _cur_win = _i
                                _t0 = _tm.time()
                                _cut = _ser[:n * (_m_c - (_i + 1))]
                                _pred = one_period_prediction_global_svd(
                                    m_hat, _m_c - (_i + 1), clf, n, _cut,
                                    g_orders=g_orders, g_b_orders=g_b_orders,
                                    g_forecaster=g_fc, g_b_forecaster=gb_fc,
                                    transform=transform, power_p=0.5, r_b=r_b, center=centering,
                                    row_mean_forecaster=center_fc,
                                    full_series=_cut,
                                )
                                gqrspm_prediction[_ctry].append(_pred)
                                _done += 1; times.append(_tm.time() - _t0)
                                _pct = _done / total * 100
                                _avg = sum(times) / len(times)
                                _rem = (total - _done) * _avg
                                _eta = f"{int(_rem//60)}m {int(_rem%60)}s" if _rem >= 60 else f"{int(_rem)+1}s"
                                _state.update({'pct': _pct, 'eta': _eta})
                            if _stop.is_set(): break
                else:
                    # UV worker
                    total = p; _done = 0
                    if model == "qspm":
                        clear_g_store()
                        qspm_prediction = []
                        for _i in range(p):
                            if _stop.is_set(): break
                            _cur_win = _i
                            _t0 = _tm.time()
                            _cut = export_vector[:n * (m - (_i + 1))]
                            _pred = one_period_prediction_classified(
                                m_hat, m - (_i + 1), clf, n, _cut,
                                g_orders=g_orders, g_b_orders=g_b_orders,
                                g_forecaster=g_fc, g_b_forecaster=gb_fc,
                                transform=transform, power_p=0.5, r_b=r_b, center=centering,
                                column_mean_order=(1, 1, 0), row_mean_order=(1, 1, 0),
                                full_series=export_vector,
                            )
                            qspm_prediction.append(_pred)
                            _done += 1; times.append(_tm.time() - _t0)
                            _pct = _done / total * 100
                            _avg = sum(times) / len(times)
                            _rem = (total - _done) * _avg
                            _eta = f"{int(_rem//60)}m {int(_rem%60)}s" if _rem >= 60 else f"{int(_rem)+1}s"
                            _state.update({'pct': _pct, 'eta': _eta})
                    elif model == "gqrspm":
                        clear_g_store()
                        gqrspm_prediction = []
                        for _i in range(p):
                            if _stop.is_set(): break
                            _cur_win = _i
                            _t0 = _tm.time()
                            _cut = export_vector[:n * (m - (_i + 1))]
                            _pred = one_period_prediction_global_svd(
                                m_hat, m - (_i + 1), clf, n, _cut,
                                g_orders=g_orders, g_b_orders=g_b_orders,
                                g_forecaster=g_fc, g_b_forecaster=gb_fc,
                                transform=transform, power_p=0.5, r_b=r_b, center=centering,
                                row_mean_forecaster=center_fc,
                                full_series=export_vector,
                            )
                            gqrspm_prediction.append(_pred)
                            _done += 1; times.append(_tm.time() - _t0)
                            _pct = _done / total * 100
                            _avg = sum(times) / len(times)
                            _rem = (total - _done) * _avg
                            _eta = f"{int(_rem//60)}m {int(_rem%60)}s" if _rem >= 60 else f"{int(_rem)+1}s"
                            _state.update({'pct': _pct, 'eta': _eta})
                if not _stop.is_set():
                    _state.update({'pct': 100.0, 'eta': 'Hotovo'})
            except Exception as _ex:
                import traceback as _tbw
                _tb_str = _tbw.format_exc()
                _ctx = f" [krajina={_cur_ctry}, okno={_cur_win}]" if _cur_ctry else ""
                print(f"[WORKER ERROR]{_ctx}")
                print(_tb_str)
                _state['error'] = f'{type(_ex).__name__}: {_ex}{_ctx}'
            finally:
                _state['running'] = False

        _thr.Thread(target=_worker, daemon=True).start()
        return 'ok'

    _co.register_callback(f'uf_get_{_uid}',   _cb_get)
    _co.register_callback(f'uf_start_{_uid}', _cb_start)
    _co.register_callback(f'uf_stop_{_uid}',  _cb_stop)

    _OUTER = ('display:inline-block;width:240px;height:8px;background:#0c0e16;'
              'border:1px solid #2e3550;border-radius:4px;overflow:hidden;vertical-align:middle')
    _FILL  = 'height:100%;background:linear-gradient(90deg,#2a4a80,#4a7fc1);transition:width 0.4s ease;width:0%'
    _ETA_S = 'font-family:Inter,sans-serif;font-size:11px;color:#5a7aaa;white-space:nowrap;margin-left:10px'
    _BTN_S = ('background:#1e2e40;color:#ffffff;border:1px solid #2e3550;border-radius:5px;'
              'font-family:Inter,sans-serif;font-size:13px;font-weight:500;'
              'width:160px;height:38px;cursor:pointer;transition:background 0.2s,color 0.2s,border-color 0.2s')
    _WRAP_S = ('background:#0c0e16;padding:14px 18px;border:1px solid #1e2438;'
               'width:780px;box-sizing:border-box;display:flex;align-items:center')

    _dsp(_HTML(f'''
<div id="uf-wrap-{_uid}" style="{_WRAP_S}">
  <button id="uf-btn-{_uid}" style="{_BTN_S}">&#9654; Spusti&#357; model</button>
  <div style="{_OUTER};margin-left:14px"><div id="uf-fill-{_uid}" style="{_FILL}"></div></div>
  <span id="uf-eta-{_uid}" style="{_ETA_S}"></span>
</div>
<script>
(function() {{
  const uid  = '{_uid}';
  const btn  = document.getElementById('uf-btn-'  + uid);
  const fill = document.getElementById('uf-fill-' + uid);
  const eta  = document.getElementById('uf-eta-'  + uid);

  async function getState() {{
    try {{
      const r   = await google.colab.kernel.invokeFunction('uf_get_' + uid, [], {{}});
      const raw = r.data['text/plain'];
      return JSON.parse(raw.slice(1, -1));
    }} catch(e) {{ return null; }}
  }}

  function applyState(s) {{
    if (!s) return;
    fill.style.width = s.pct + '%';
    eta.textContent  = s.error ? 'Chyba: ' + s.error : s.eta;
    if (s.running) {{
      btn.textContent       = '\u23f9 Zastavi\u0165';
      btn.style.background  = '#5a1a1a';
      btn.style.color       = '#e05a5a';
      btn.style.borderColor = '#7a2a2a';
    }} else {{
      btn.textContent       = '\u25b6 Spusti\u0165 model';
      btn.style.background  = '#1e2e40';
      btn.style.color       = '#ffffff';
      btn.style.borderColor = '#2e3550';
    }}
  }}

  let _polling = false;
  function startPolling() {{
    if (window._uf_poll) clearInterval(window._uf_poll);
    window._uf_poll = setInterval(async () => {{
      if (_polling) return;
      _polling = true;
      try {{
        const s = await getState();
        applyState(s);
        if (s && !s.running) {{ clearInterval(window._uf_poll); window._uf_poll = null; }}
      }} finally {{ _polling = false; }}
    }}, 400);
  }}

  btn.addEventListener('click', async function() {{
    const s = await getState();
    if (s && s.running) {{
      google.colab.kernel.invokeFunction('uf_stop_' + uid, [], {{}});
      eta.textContent = 'Zastavujem...';
    }} else {{
      var _p={{}};
      document.querySelectorAll('input[data-name]').forEach(function(el){{_p[el.dataset.name]=el.value;}});
      document.querySelectorAll('.uf-dd[data-name]').forEach(function(dd){{_p[dd.dataset.name]=dd.dataset.current||(dd.querySelector('.uf-dd-opt')?dd.querySelector('.uf-dd-opt').dataset.val:'');}});
      await google.colab.kernel.invokeFunction('uf_start_' + uid, [JSON.stringify(_p)], {{}});
      applyState(await getState());
      startPolling();
    }}
  }});
}})();
</script>
'''))

def _uf_show_results():
    try:
        _ds = DATASET
    except NameError:
        from IPython.display import display, HTML
        display(HTML('<div style="font-family:Inter,sans-serif;color:#5a6e8a;font-size:13px;padding:6px 0">Spusti najprv <b>_uf_load_data()</b>.</div>'))
        return
    if _ds in ('cz10c', 'cz30c'):
        _uf_show_results_mv()
    else:
        _uf_show_results_uv()


def _uf_show_results_uv():
    import numpy as _npR
    from IPython.display import display, HTML

    def _lrR(a, b):
        try:
            fa, fb = float(a), float(b)
            return float(_npR.log(fa / fb)) if fa > 0 and fb > 0 else float('nan')
        except Exception: return float('nan')

    def _clrR(v):
        try:
            fv = float(v)
            if _npR.isnan(fv): return '#3a4a5a'
            return '#6adb8a' if fv < 0 else '#e05a5a'
        except Exception: return '#3a4a5a'

    def _fmtR(v):
        try:
            fv = float(v)
            return '–' if _npR.isnan(fv) else f'{fv:+.3f}'
        except Exception: return '–'

    def _mseR(yt, yp):
        yt, yp = _npR.asarray(yt, float), _npR.asarray(yp, float)
        return float(_npR.mean((yt - yp) ** 2))

    def _mcleR(yt, yp, tol=0.05):
        yt, yp = _npR.asarray(yt, float), _npR.asarray(yp, float)
        denom = _npR.where(_npR.abs(yt) > 0, tol * _npR.abs(yt), _npR.nan)
        return float(_npR.nanmean(_npR.minimum(_npR.abs(yt - yp) / denom, 1.0)))

    def _maseR(act, pred, train, s=12):
        act, pred, train = _npR.asarray(act, float), _npR.asarray(pred, float), _npR.asarray(train, float)
        if len(train) <= s: return float('nan')
        D = _npR.mean(_npR.abs(train[s:] - train[:-s]))
        if D == 0: return float('nan')
        return float(_npR.mean(_npR.abs(act - pred)) / D)

    _TH  = ('padding:6px 14px;text-align:right;font-family:Inter,sans-serif;'
            'font-size:12px;color:#7aaae8;font-weight:600;border-bottom:1px solid #2e3550')
    _TH0 = _TH.replace('text-align:right', 'text-align:left')
    _TD  = ('padding:5px 14px;text-align:right;font-family:Inter,sans-serif;'
            'font-size:12px;color:#c8d4e8')
    _TD0 = _TD.replace('text-align:right', 'text-align:left')

    def _cellR(v, bold=False):
        _fw = 'font-weight:600;' if bold else ''
        return f'<td style="{_TD};{_fw}color:{_clrR(v)}">{_fmtR(v)}</td>'

    def _render_table(rows, lr_mse_c, lr_mas_c, lr_mcle_c, label):
        _html_r = ''
        for _wi, _lm, _la, _lc in rows:
            _html_r += (f'<tr><td style="{_TD0}">{_wi}</td>'
                        f'{_cellR(_lm)}{_cellR(_la)}{_cellR(_lc)}</tr>')
        _html_r += (
            f'<tr style="border-top:1px solid #2e3550">'
            f'<td style="{_TD0};font-weight:600;color:#ffffff">Priemer</td>'
            f'{_cellR(lr_mse_c, True)}{_cellR(lr_mas_c, True)}{_cellR(lr_mcle_c, True)}</tr>'
        )
        display(HTML(
            f'<div style="font-family:Inter,sans-serif;margin:12px 0 6px 0">'
            f'<span style="font-size:13px;font-weight:600;color:#7aaae8">'
            f'{label} – log(Model / SARIMA)</span>'
            f'<div style="font-size:11px;color:#5a6e8a;margin-top:3px">'
            f'záporné&nbsp;=&nbsp;lepšie ako SARIMA'
            f'&nbsp;|&nbsp;MSE, MASE (s=12), MCLE (tol=5%)'
            f'</div></div>'
            f'<div style="background:#0c0e16;border:1px solid #1e2438;border-radius:6px;'
            f'overflow:hidden;display:inline-block;margin:4px 0 8px 0">'
            f'<table style="border-collapse:collapse;min-width:340px">'
            f'<thead><tr>'
            f'<th style="{_TH0}">Krajina</th>'
            f'<th style="{_TH}">MSE</th>'
            f'<th style="{_TH}">MASE</th>'
            f'<th style="{_TH}">MCLE</th>'
            f'</tr></thead>'
            f'<tbody>{_html_r}</tbody>'
            f'</table></div>'
        ))
    try:
        _model_val = w_model.value
        _ap = qspm_prediction if _model_val == 'qspm' else gqrspm_prediction
        if not isinstance(_ap, list):
            raise TypeError('Predikcia nie je list - zmenil sa dataset? Spusti model znova.')
    except NameError:
        display(HTML('<div style="font-family:Inter,sans-serif;color:#5a6e8a;font-size:13px;padding:6px 0">Predikcia ešte nebola spustená - klikni <b>&#9654; Spustiť model</b>.</div>'))
        return
    except Exception as _ex:
        display(HTML(f'<div style="font-family:Inter,sans-serif;color:#e05a5a;font-size:13px;padding:6px 0">{_ex}</div>'))
        return

    _label = 'QS-PM' if _model_val == 'qspm' else 'GQRS-PM'
    _sar = sarima_prediction
    _ev  = export_vector
    _n   = n
    _m   = m

    _rows = []
    _all_tr, _all_pr, _all_sr = [], [], []
    _mase_m_l, _mase_s_l = [], []

    for _wi in range(len(_ap)):
        _tr = _npR.asarray(_ev[_n * (_m - _wi - 1) : _n * (_m - _wi)], float)
        _pr = _npR.asarray(_ap[_wi], float)
        _tn = _npR.asarray(_ev[:_n * (_m - _wi - 1)], float)
        _all_tr.append(_tr); _all_pr.append(_pr)
        _mse_m = _mseR(_tr,_pr); _mcle_m = _mcleR(_tr,_pr); _mas_m = _maseR(_tr,_pr,_tn)
        _mse_s = _mcle_s = _mas_s = float('nan')
        if _sar is not None and _wi < len(_sar):
            _sp = _npR.asarray(_sar[_wi], float)
            _all_sr.append(_sp)
            _mse_s = _mseR(_tr,_sp); _mcle_s = _mcleR(_tr,_sp); _mas_s = _maseR(_tr,_sp,_tn)
            if not _npR.isnan(_mas_s): _mase_s_l.append(_mas_s)
        else:
            _all_sr.append(None)
        if not _npR.isnan(_mas_m): _mase_m_l.append(_mas_m)
        _rows.append((_wi+1, _lrR(_mse_m,_mse_s), _lrR(_mas_m,_mas_s), _lrR(_mcle_m,_mcle_s)))

    _ct = _npR.concatenate(_all_tr)
    _cp = _npR.concatenate(_all_pr)
    _lr_mse_c = _lr_mas_c = _lr_mcle_c = float('nan')
    if _all_sr and all(_s is not None for _s in _all_sr):
        _cs = _npR.concatenate(_all_sr)
        _lr_mse_c  = _lrR(_mseR(_ct, _cp), _mseR(_ct, _cs))
        _lr_mcle_c = _lrR(_mcleR(_ct, _cp), _mcleR(_ct, _cs))
        _mse_mc = float(_npR.nanmean(_mase_m_l)) if _mase_m_l else float('nan')
        _mse_sc = float(_npR.nanmean(_mase_s_l)) if _mase_s_l else float('nan')
        _lr_mas_c = _lrR(_mse_mc, _mse_sc)

    _render_table(_rows, _lr_mse_c, _lr_mas_c, _lr_mcle_c, _label)

    # Walk-forward plot
    import matplotlib.pyplot as _plt_wf
    _fig, _ax = _plt_wf.subplots(figsize=(14, 4))
    _ax.plot(range(len(_ev)), _ev, color='#2c3e50', linewidth=1.4, label='Skutočná séria', zorder=2)
    for _wi in range(len(_all_pr)):
        _ts = _n * (_m - _wi - 1)
        _te = _n * (_m - _wi)
        _ax.axvspan(_ts, _te, alpha=0.08, color='gray', zorder=1)
        _ax.plot(range(_ts, _te), _all_pr[_wi], color='#e74c3c', linewidth=1.8, linestyle='--', zorder=3,
                 label=_label if _wi == 0 else '_nolegend_')
        if _all_sr[_wi] is not None:
            _ax.plot(range(_ts, _te), _all_sr[_wi], color='#3498db', linewidth=1.8, linestyle='-.', zorder=3,
                     label='SARIMA' if _wi == 0 else '_nolegend_')
    try:
        _xt = list(range(0, len(_ev), _n))
        _ax.set_xticks(_xt)
        _ax.set_xticklabels([str(start_year + i) for i in range(len(_xt))], rotation=45, ha='right', fontsize=9)
    except Exception: pass
    _ax.set_title(f'Walk-forward predikcia – {_label}', fontsize=12, fontweight='bold')
    _ax.set_xlabel('Rok', fontsize=10); _ax.set_ylabel('Hodnota', fontsize=10)
    _ax.legend(fontsize=9); _ax.grid(True, alpha=0.3)
    _ax.spines[['top', 'right']].set_visible(False)
    _plt_wf.tight_layout(); _plt_wf.show()

    # G-série a mean-série: predikcia vs. empiria
    try:
        if _g_store_list:
            display(HTML(
                '<div style="font-family:Inter,sans-serif;margin:20px 0 6px 0">'
                '<span style="font-size:13px;font-weight:600;color:#7aaae8">'
                'G-série a mean-série – predikcia vs. empiria'
                '</span>'
                '<div style="font-size:11px;color:#5a6e8a;margin-top:3px">'
                'Regulárne (g) a neregulárne (g_b) koeficienty pre každé walk-forward okno. '
                'Hviezda&nbsp;=&nbsp;predikcia, kosoštvorec&nbsp;=&nbsp;empirická hodnota.'
                '</div></div>'
            ))
            plot_stored_g_series(style='standard', centering=w_center.value)
    except NameError:
        pass


def _uf_show_results_mv():
    import numpy as _npR
    from IPython.display import display, HTML

    def _lrR(a, b):
        try:
            fa, fb = float(a), float(b)
            return float(_npR.log(fa / fb)) if fa > 0 and fb > 0 else float('nan')
        except Exception: return float('nan')

    def _clrR(v):
        try:
            fv = float(v)
            if _npR.isnan(fv): return '#3a4a5a'
            return '#6adb8a' if fv < 0 else '#e05a5a'
        except Exception: return '#3a4a5a'

    def _fmtR(v):
        try:
            fv = float(v)
            return '–' if _npR.isnan(fv) else f'{fv:+.3f}'
        except Exception: return '–'

    def _mseR(yt, yp):
        yt, yp = _npR.asarray(yt, float), _npR.asarray(yp, float)
        return float(_npR.mean((yt - yp) ** 2))

    def _mcleR(yt, yp, tol=0.05):
        yt, yp = _npR.asarray(yt, float), _npR.asarray(yp, float)
        denom = _npR.where(_npR.abs(yt) > 0, tol * _npR.abs(yt), _npR.nan)
        return float(_npR.nanmean(_npR.minimum(_npR.abs(yt - yp) / denom, 1.0)))

    def _maseR(act, pred, train, s=12):
        act, pred, train = _npR.asarray(act, float), _npR.asarray(pred, float), _npR.asarray(train, float)
        if len(train) <= s: return float('nan')
        D = _npR.mean(_npR.abs(train[s:] - train[:-s]))
        if D == 0: return float('nan')
        return float(_npR.mean(_npR.abs(act - pred)) / D)

    _TH  = ('padding:6px 14px;text-align:right;font-family:Inter,sans-serif;'
            'font-size:12px;color:#7aaae8;font-weight:600;border-bottom:1px solid #2e3550')
    _TH0 = _TH.replace('text-align:right', 'text-align:left')
    _TD  = ('padding:5px 14px;text-align:right;font-family:Inter,sans-serif;'
            'font-size:12px;color:#c8d4e8')
    _TD0 = _TD.replace('text-align:right', 'text-align:left')

    def _cellR(v, bold=False):
        _fw = 'font-weight:600;' if bold else ''
        return f'<td style="{_TD};{_fw}color:{_clrR(v)}">{_fmtR(v)}</td>'

    def _render_table(rows, lr_mse_c, lr_mas_c, lr_mcle_c, label):
        _html_r = ''
        for _wi, _lm, _la, _lc in rows:
            _html_r += (f'<tr><td style="{_TD0}">{_wi}</td>'
                        f'{_cellR(_lm)}{_cellR(_la)}{_cellR(_lc)}</tr>')
        _html_r += (
            f'<tr style="border-top:1px solid #2e3550">'
            f'<td style="{_TD0};font-weight:600;color:#ffffff">Priemer</td>'
            f'{_cellR(lr_mse_c, True)}{_cellR(lr_mas_c, True)}{_cellR(lr_mcle_c, True)}</tr>'
        )
        display(HTML(
            f'<div style="font-family:Inter,sans-serif;margin:12px 0 6px 0">'
            f'<span style="font-size:13px;font-weight:600;color:#7aaae8">'
            f'{label} – log(Model / SARIMA)</span>'
            f'<div style="font-size:11px;color:#5a6e8a;margin-top:3px">'
            f'záporné&nbsp;=&nbsp;lepšie ako SARIMA'
            f'&nbsp;|&nbsp;MSE, MASE (s=12), MCLE (tol=5%)'
            f'</div></div>'
            f'<div style="background:#0c0e16;border:1px solid #1e2438;border-radius:6px;'
            f'overflow:hidden;display:inline-block;margin:4px 0 8px 0">'
            f'<table style="border-collapse:collapse;min-width:340px">'
            f'<thead><tr>'
            f'<th style="{_TH0}">Krajina</th>'
            f'<th style="{_TH}">MSE</th>'
            f'<th style="{_TH}">MASE</th>'
            f'<th style="{_TH}">MCLE</th>'
            f'</tr></thead>'
            f'<tbody>{_html_r}</tbody>'
            f'</table></div>'
        ))
    try:
        _model_val = w_model.value
        _ap = qspm_prediction if _model_val == 'qspm' else gqrspm_prediction
        if not isinstance(_ap, dict):
            raise TypeError('Predikcia nie je dict - zmenil sa dataset? Spusti model znova pre CZ.')
        _ed = export_dict
        if _ed is None:
            raise RuntimeError('export_dict je None - spusti _uf_load_data() pre CZ dataset.')
    except NameError:
        display(HTML('<div style="font-family:Inter,sans-serif;color:#5a6e8a;font-size:13px;padding:6px 0">Predikcia ešte nebola spustená - klikni <b>&#9654; Spustiť model</b>.</div>'))
        return
    except Exception as _ex:
        display(HTML(f'<div style="font-family:Inter,sans-serif;color:#e05a5a;font-size:13px;padding:6px 0">{_ex}</div>'))
        return

    _model_label = 'QS-PM' if _model_val == 'qspm' else 'GQRS-PM'
    _sar = sarima_prediction
    _n   = n

    _ctrs  = list(_ap.keys())
    _n_win = min(len(v) for v in _ap.values())
    _label = _model_label + f' (priemer cez {_n_win} okien)'
    _nan   = float('nan')
    _rows  = []
    _all_lm, _all_la, _all_lc = [], [], []

    for _ctry in _ctrs:
        _ser = _ed[_ctry]
        _m_c = len(_ser) // _n
        _mm, _am, _cm, _ms, _as2, _cs2 = [], [], [], [], [], []
        for _wi in range(_n_win):
            _tr = _npR.asarray(_ser[_n*(_m_c-_wi-1):_n*(_m_c-_wi)], float)
            _tn = _npR.asarray(_ser[:_n*(_m_c-_wi-1)], float)
            _pr = _npR.asarray(_ap[_ctry][_wi], float)
            _mm.append(_mseR(_tr,_pr)); _am.append(_maseR(_tr,_pr,_tn)); _cm.append(_mcleR(_tr,_pr))
            if _sar and _ctry in _sar and _wi < len(_sar[_ctry]):
                _sp = _npR.asarray(_sar[_ctry][_wi], float)
                _ms.append(_mseR(_tr,_sp)); _as2.append(_maseR(_tr,_sp,_tn)); _cs2.append(_mcleR(_tr,_sp))
        _mse_m  = float(_npR.nanmean(_mm)) if _mm else _nan
        _mas_m  = float(_npR.nanmean([x for x in _am if not _npR.isnan(x)])) if any(not _npR.isnan(x) for x in _am) else _nan
        _mcle_m = float(_npR.nanmean(_cm)) if _cm else _nan
        _mse_s  = float(_npR.nanmean(_ms))  if _ms  else _nan
        _mas_s  = float(_npR.nanmean([x for x in _as2 if not _npR.isnan(x)])) if any(not _npR.isnan(x) for x in _as2) else _nan
        _mcle_s = float(_npR.nanmean(_cs2)) if _cs2 else _nan
        _lm = _lrR(_mse_m, _mse_s); _la = _lrR(_mas_m, _mas_s); _lc = _lrR(_mcle_m, _mcle_s)
        _rows.append((_ctry, _lm, _la, _lc))
        if not _npR.isnan(_lm): _all_lm.append(_lm)
        if not _npR.isnan(_la): _all_la.append(_la)
        if not _npR.isnan(_lc): _all_lc.append(_lc)

    _lr_mse_c  = float(_npR.nanmean(_all_lm)) if _all_lm else _nan
    _lr_mas_c  = float(_npR.nanmean(_all_la)) if _all_la else _nan
    _lr_mcle_c = float(_npR.nanmean(_all_lc)) if _all_lc else _nan

    _render_table(_rows, _lr_mse_c, _lr_mas_c, _lr_mcle_c, _label)


def _uf_show_diagnostics():
    from IPython.display import display, HTML
    import os as _os2, pandas as _pd2

    def _hdr(title, desc=''):
        display(HTML(
            '<div style="font-family:Inter,sans-serif;margin:28px 0 6px 0">' +
            '<span style="font-size:13px;font-weight:600;color:#7aaae8">' + title + '</span>' +
            ('<div style="font-size:11.5px;color:#5a6e8a;margin-top:3px">' + desc + '</div>' if desc else '') +
            '</div>'
        ))

    try:
        _ds = DATASET
    except NameError:
        display(HTML('<div style="font-family:Inter,sans-serif;color:#5a6e8a;font-size:13px;padding:6px 0">Spusti najprv <b>_uf_load_data()</b>.</div>'))
        return

    _mhat = int(w_mhat.value)
    _r    = int(w_r.value)
    _p    = int(w_p_win.value)

    if _ds == 'sk':
        _hdr('Vývoj singulárneho spektra (3D)',
             'Evolúcia singulárnych hodnôt matice X naprieč kĺzavými oknami.')
        plot_singular_value_spectrum_3d(
            export_vector, m_hat=_mhat, m=m, n=n,
            start_year=start_year, center=w_center.value,
            testset_size=6,
        )

        _hdr('Neregulárna časť – singulárne spektrum (3D)',
             'Singulárne hodnoty B-priestoru naprieč kĺzavými oknami.')
        plot_nonregular_spectrum_3d(
            export_vector, m_hat=_mhat, m=m, n=n,
            r=_r, transform=w_transform.value,
            start_year=start_year, center=w_center.value,
            testset_size=6,
        )

        _hdr('Autokorelácia (ACF)',
             'Autokorelačná funkcia časového radu.')
        plot_autocorrelation(export_vector, n=12, lags=48, show_pacf=False)

        _hdr('Singulárne komponenty – sezónne vzory (4×3)',
             'Pravé singulárne vektory (Vt) z SVD tréningových dát. '
             'Červená = kladné, modrá = záporné hodnoty. '
             'Energetický podiel v záhlaví každého panela.')
        import matplotlib.pyplot as _plt_sc
        import numpy as _np_sc

        def _pca_sc(data):
            yr = len(data) // n
            X  = _np_sc.asarray(data[:yr * n], dtype=float).reshape(yr, n)
            _ctr = w_center.value
            if _ctr == 'column':
                Xc = X - X.mean(axis=0)
            elif _ctr == 'row':
                Xc = X - X.mean(axis=1)[:, _np_sc.newaxis]
            elif _ctr == 'double':
                _cm = X.mean(axis=0); _rm = X.mean(axis=1); _gm = float(X.mean())
                Xc  = X - _cm - _rm[:, _np_sc.newaxis] + _gm
            else:
                Xc = X.copy()
            U_sc, s_sc, Vt_sc = _np_sc.linalg.svd(Xc, full_matrices=False)
            _mi = _np_sc.argmax(_np_sc.abs(Vt_sc), axis=1)
            _sg = _np_sc.sign(Vt_sc[_np_sc.arange(len(_mi)), _mi])
            Vt_sc *= _sg[:, _np_sc.newaxis]
            U_sc  *= _sg[_np_sc.newaxis, :]
            expl_sc = s_sc**2 / _np_sc.dot(s_sc, s_sc)
            return Vt_sc, expl_sc, s_sc, U_sc

        _n_grid_rows, _n_grid_cols = 4, 3
        _Vt_sc, _expl_sc, _s_sc, _U_sc = _pca_sc(export_vector[:n * m])
        _nc_sc = min(12, _Vt_sc.shape[0])
        _months_sc = _np_sc.arange(1, n + 1)

        _fig_sc, _axes_sc = _plt_sc.subplots(
            _n_grid_rows, _n_grid_cols,
            figsize=(_n_grid_cols * 3.2, _n_grid_rows * 2.5),
            squeeze=False)

        for _i_sc in range(_nc_sc):
            _ri, _ci = _i_sc // _n_grid_cols, _i_sc % _n_grid_cols
            _ax_sc = _axes_sc[_ri, _ci]
            _comp  = _Vt_sc[_i_sc]
            _cols_sc = ['#c0392b' if v >= 0 else '#2980b9' for v in _comp]
            _ax_sc.bar(_months_sc, _comp, color=_cols_sc, alpha=0.85, width=0.75)
            _ax_sc.axhline(0, color='#444', linewidth=0.6)
            _ax_sc.set_title(
                f'$\mathrm{{SV}}_{{{_i_sc + 1}}}$  ({_expl_sc[_i_sc] * 100:.4f}%)',
                fontsize=10)
            _ax_sc.set_xticks(_months_sc)
            _ax_sc.set_xticklabels(list(_months_sc), fontsize=8)
            _ax_sc.tick_params(axis='y', labelsize=8)
            if _ri == _n_grid_rows - 1 or _i_sc == _nc_sc - 1:
                _ax_sc.set_xlabel('Mesiac', fontsize=9)
            _ax_sc.spines[['top', 'right']].set_visible(False)
            _ax_sc.grid(True, axis='y', alpha=0.18, linestyle='--')

        for _j_sc in range(_nc_sc, _n_grid_rows * _n_grid_cols):
            _axes_sc[_j_sc // _n_grid_cols, _j_sc % _n_grid_cols].set_visible(False)

        _fig_sc.suptitle(
            f'Singulárne vektory – tréningové dáta (m={m} rokov)',
            fontsize=10, y=1.01)
        _fig_sc.tight_layout()
        _plt_sc.show()

        _hdr('Ročné zastúpenie SV komponentov (U matica)',
             'Stĺpce U matice ukazujú, ako silno je každý rok zastúpený v danom SV komponente.')
        _yr_start_sc = globals().get('start_year', 0)
        _yr_sc = _U_sc.shape[0]
        _nc_u  = min(12, _U_sc.shape[1])
        _years_sc = [_yr_start_sc + k for k in range(_yr_sc)]

        _fig_u, _axes_u = _plt_sc.subplots(
            _n_grid_rows, _n_grid_cols,
            figsize=(_n_grid_cols * 3.2, _n_grid_rows * 2.5),
            squeeze=False)

        for _i_u in range(_nc_u):
            _ax_u = _axes_u[_i_u // _n_grid_cols, _i_u % _n_grid_cols]
            _vals_u = _U_sc[:, _i_u]
            _cols_u = ['#e52222' if v >= 0 else '#2980b9' for v in _vals_u]
            _ax_u.bar(_years_sc, _vals_u, color=_cols_u, alpha=0.85, width=0.75)
            _ax_u.axhline(0, color='#444', linewidth=0.6)
            _ax_u.set_title(
                f'$\mathrm{{SV}}_{{{_i_u + 1}}}$  ({_expl_sc[_i_u] * 100:.4f}%)',
                fontsize=10)
            _step_u = max(1, _yr_sc // 8)
            _ax_u.set_xticks(_years_sc[::_step_u])
            _ax_u.set_xticklabels(
                [str(y) for y in _years_sc[::_step_u]],
                rotation=45, ha='right', fontsize=8)
            _ax_u.tick_params(axis='y', labelsize=8)
            _ax_u.spines[['top', 'right']].set_visible(False)
            _ax_u.grid(True, axis='y', alpha=0.18, linestyle='--')

        for _j_u in range(_nc_u, _n_grid_rows * _n_grid_cols):
            _axes_u[_j_u // _n_grid_cols, _j_u % _n_grid_cols].set_visible(False)

        _fig_u.suptitle(
            f'Ročné zastúpenie SV – tréningové dáta (m={m} rokov)',
            fontsize=10, y=1.01)
        _fig_u.tight_layout()
        _plt_sc.show()

    elif _ds in ('cz10c', 'cz30c'):
        _hdr('Energia prvého SVD komponentu',
             'Histogram podielu energie prvého komponentu – krajiny vybraného CZ datasetu.')
        _hp_cz = _DATASETS[_ds]
        if _os2.path.exists(_hp_cz):
            plot_spectrum_histogram(_hp_cz, sk_series=export_vector, ap_series=None,
                                    n=12, m_hat=_mhat, p=_p)
        else:
            display(HTML(f'<p style="color:#5a6e8a;font-size:12px;font-family:Inter,sans-serif">Súbor {_hp_cz} nenájdený – histogram preskočený</p>'))

In [66]:
def _run_active_model():
    import warnings, time as _time
    model     = w_model.value
    m_hat     = int(w_mhat.value)
    p         = int(w_p_win.value)
    r         = int(w_r.value)
    r_b       = int(w_rb.value)
    g_orders   = [(int(w_gp.value), int(w_gd.value), int(w_gq.value))]
    g_b_orders = [(int(w_gbp.value), int(w_gbd.value), int(w_gbq.value))]
    g_fc      = w_fc_g.value
    gb_fc     = w_fc_gb.value
    transform = w_transform.value
    centering = w_center.value

    classifications = {i: 'r_n' for i in range(r)}
    for _i in range(r, m_hat):
        classifications[_i] = 'r_b'
    classifications['r']   = r
    classifications['r_b'] = r_b

    total_windows = p  # 1 dataset - p walk-forward windows
    _win_times = []

    print(f'[_run_active_model] model={model!r}  p={p}  m_hat={m_hat}  r={r}  r_b={r_b}', flush=True)

    if model == 'qspm':
        clear_g_store()
        global qspm_prediction
        qspm_prediction = []
        for _i in range(p):
            print(f'[QS-PM] okno {_i+1}/{total_windows} začína', flush=True)
            _t0 = _time.time()
            _cut = export_vector[:n * (m - (_i + 1))]
            qspm_prediction.append(one_period_prediction_classified(
                m_hat, m - (_i + 1), classifications, n, _cut,
                g_orders=g_orders, g_b_orders=g_b_orders,
                g_forecaster=g_fc, g_b_forecaster=gb_fc,
                transform=transform, power_p=0.5,
                r_b=r_b, center=centering,
                column_mean_order=(1, 1, 0),
                row_mean_order=(1, 1, 0),
                full_series=export_vector,
            ))
            _win_times.append(_time.time() - _t0)
            _avg_ms = int(sum(_win_times) / len(_win_times) * 1000)
            print(f'[QS-PM] okno {_i+1}/{total_windows} hotovo ({_win_times[-1]:.1f}s)', flush=True)
            print(f'PROGRESS:{_i+1}:{total_windows}:{_avg_ms}', flush=True)
        print(f'[QS-PM] všetky okná hotovo', flush=True)

    elif model == 'gqrspm':
        clear_g_store()
        global gqrspm_prediction
        gqrspm_prediction = []
        for _i in range(p):
            print(f'[GQRS-PM] okno {_i+1}/{total_windows} začína', flush=True)
            _t0 = _time.time()
            _cut = export_vector[:n * (m - (_i + 1))]
            gqrspm_prediction.append(one_period_prediction_global_svd(
                m_hat, m - (_i + 1), classifications, n, _cut,
                g_orders=g_orders, g_b_orders=g_b_orders,
                g_forecaster=g_fc, g_b_forecaster=gb_fc,
                transform=transform, power_p=0.5,
                r_b=r_b, center=centering,
                row_mean_order=(1, 1, 0),
                row_mean_forecaster=None,
                full_series=_cut,
            ))
            _win_times.append(_time.time() - _t0)
            _avg_ms = int(sum(_win_times) / len(_win_times) * 1000)
            print(f'[GQRS-PM] okno {_i+1}/{total_windows} hotovo ({_win_times[-1]:.1f}s)', flush=True)
            print(f'PROGRESS:{_i+1}:{total_windows}:{_avg_ms}', flush=True)
        print(f'[GQRS-PM] všetky okná hotovo', flush=True)

    else:
        print(f'Neznámy model: {model!r}')


In [67]:
%matplotlib inline
import numpy as np
import pandas as pd
import warnings
import pickle, os
from scipy.linalg import svd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima
from scipy import stats as _stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pyplot as _plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from IPython.display import display as _disp
warnings.filterwarnings('ignore')


In [68]:
def process_vector_to_svd(input_vector, center="none"):
    segment_length = 12
    n = len(input_vector)
    num_segments = n // segment_length
    data_matrix = np.array(input_vector[:num_segments * segment_length]).reshape(num_segments, segment_length)
    grand_mean = 0.0
    if center == "column":
        X_mean = data_matrix.mean(axis=0)
        data_matrix = data_matrix - X_mean
    elif center == "row":
        X_mean = data_matrix.mean(axis=1)
        data_matrix = data_matrix - X_mean[:, np.newaxis]
    elif center == "double":
        col_mean   = data_matrix.mean(axis=0)
        row_mean_v = data_matrix.mean(axis=1)
        grand_mean = float(data_matrix.mean())
        data_matrix = data_matrix - col_mean - row_mean_v[:, np.newaxis] + grand_mean
        X_mean = col_mean
    else:
        X_mean = np.zeros(segment_length)
    U, s, Vt = svd(data_matrix, full_matrices=False)
    # Stabilizácia znamienka: dominantný prvok každého riadka Vt je kladný
    max_abs = np.argmax(np.abs(Vt), axis=1)
    signs   = np.sign(Vt[np.arange(len(max_abs)), max_abs])
    Vt *= signs[:, np.newaxis]
    U  *= signs[np.newaxis, :]
    return {"original_matrix": data_matrix, "U": U, "s": s, "Vt": Vt,
            "segments_used": num_segments, "X_mean": X_mean, "grand_mean": grand_mean}

def _get_indices(classifications):
    """Extract sorted r_n, r_b, and redundant component index lists."""
    r_n = sorted(k for k, v in classifications.items()
                 if isinstance(k, int) and v == "r_n")
    r_b = sorted(k for k, v in classifications.items()
                 if isinstance(k, int) and v == "r_b")
    redundant = sorted(k for k, v in classifications.items()
                       if isinstance(k, int) and v == "redundant")
    return r_n, r_b, redundant

def _make_transform(transform, power_p, epsilon=1e-10):
    """Returns a callable A_N -> B for the chosen nonlinear transform."""
    if transform == "log":
        return lambda A, c: np.log(np.maximum(A + c, epsilon))
    elif transform == "sqrt":
        return lambda A, c: np.sqrt(np.maximum(A + c, 0.0))
    elif transform == "cbrt":
        return lambda A, c: np.cbrt(A + c)
    elif transform == "power":
        return lambda A, c: np.sign(A + c) * np.abs(A + c) ** power_p
    else:
        raise ValueError(f"Neznama transformacia: {transform!r}. "
                         f"Dostupne: log, sqrt, cbrt, power")



def reconstruct_svd_excluding_top_k(input_vector, top_k=1):
    res = process_vector_to_svd(input_vector)
    s_mod = np.copy(res['s'])
    s_mod[:top_k] = 0
    return (res['U'] @ np.diag(s_mod) @ res['Vt']).flatten(), res

def reconstruct_svd_only_top_k(input_vector, top_k):
    res = process_vector_to_svd(input_vector)
    s_sel = np.zeros_like(res['s'])
    s_sel[:top_k] = res['s'][:top_k]
    return (res['U'] @ np.diag(s_sel) @ res['Vt']).flatten(), res



import json as _json, os as _os

_clf_path = 'data/classifications.json'

def _load_classifications(path=_clf_path):
    """Load classifications from JSON; restores integer keys."""
    if not _os.path.exists(path):
        return None
    with open(path, encoding='utf-8') as _f:
        raw = _json.load(_f)
    return {int(k) if k.lstrip('-').isdigit() else k: v for k, v in raw.items()}




def _make_transform(transform, power_p, epsilon=1e-10):
    """Vráti callable A_N -> B pre zvolenú nelineárnu transformáciu."""
    if transform == 'log':
        return lambda A, c: np.log(np.maximum(A + c, epsilon))
    elif transform == 'sqrt':
        return lambda A, c: np.sqrt(np.maximum(A + c, 0.0))
    elif transform == 'cbrt':
        return lambda A, c: np.cbrt(A + c)
    elif transform == 'power':
        return lambda A, c: np.sign(A + c) * np.abs(A + c) ** power_p
    else:
        raise ValueError(f'Neznama transformacia: {transform!r}. '
                         f'Dostupne: log, sqrt, cbrt, power')

def compute_g(svd_results, r):
    U, s = svd_results['U'], svd_results['s']
    return [float(U[-1, i] * s[i]) for i in range(r)]

def compute_g_b(svd_results, r, r_b, epsilon=1e-10,
                transform='log', power_p=0.5):
    U, s, Vt = svd_results['U'], svd_results['s'], svd_results['Vt']
    p = min(U.shape[0], Vt.shape[1])

    if r >= p:
        raise ValueError(f"r ({r}) must be less than p ({p})")
    if r_b > (p - r):
        raise ValueError(f"r_b ({r_b}) must be <= p-r ({p-r})")

    A_N = sum(s[i] * U[:, i:i+1] @ Vt[i:i+1, :] for i in range(r, p))
    min_AN = np.min(A_N)
    c = (-min_AN + epsilon) if min_AN <= 0 else epsilon
    f_tr = _make_transform(transform, power_p, epsilon)
    B = f_tr(A_N, c)

    res = process_vector_to_svd(B.flatten())
    g_b = [res['U'][-1, i] * res['s'][i] for i in range(r_b)]
    return np.array(g_b), res['Vt'], c

def create_data_window_vector(time_series, k, period_length, window_size):
    start_idx = (k - window_size) * period_length
    end_idx = k * period_length
    if start_idx < 0 or end_idx > len(time_series):
        raise ValueError("Window out of range")
    return time_series[start_idx:end_idx]

def create_G_matrices(window_size, m, r, r_b, time_series, period_length,
                      transform='log', power_p=0.5):
    G_matrix, G_b_matrix, c_list = [], [], []
    svd_results = None
    for k in range(window_size, m):
        window = create_data_window_vector(time_series, k, period_length, window_size)
        svd_results = process_vector_to_svd(window)
        G_matrix.append(compute_g(svd_results, r))
        g_b, Vt_b, c = compute_g_b(svd_results, r, r_b,
                                   transform=transform, power_p=power_p)
        G_b_matrix.append(g_b)
        c_list.append(c)
    return np.array(G_matrix), np.array(G_b_matrix), svd_results['Vt'], Vt_b, c_list



def _plot_x_mean_series(X_mean_matrix, pred_X_mean, emp_X_mean, prefix=''):
    """Kompaktná sieť 4x3 časových radov X_mean, jeden panel na mesiac."""
    import matplotlib.pyplot as plt
    Xm = np.asarray(X_mean_matrix, dtype=float)
    if Xm.ndim != 2 or Xm.shape[1] == 0:
        return
    T, n_months = Xm.shape
    n_cols = 4
    n_rows = (n_months + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.2, n_rows * 2.4),
                             squeeze=False)
    for j in range(n_months):
        ax = axes[j // n_cols, j % n_cols]
        series = Xm[:, j]
        ax.plot(np.arange(T), series, marker='o', markersize=3,
                linewidth=1.0, color='steelblue')
        pv = ev = None
        if pred_X_mean is not None:
            pv = float(np.asarray(pred_X_mean)[j])
            ax.plot([T - 1, T], [series[-1], pv], color='red',
                    linewidth=1.0, linestyle='--')
            ax.scatter([T], [pv], color='red', s=45, zorder=5, marker='*')
        if emp_X_mean is not None:
            ev = float(np.asarray(emp_X_mean)[j])
            ax.scatter([T], [ev], color='green', s=50, zorder=6, marker='D')
            if pv is not None:
                ax.plot([T, T], [pv, ev], color='grey',
                        linewidth=0.8, linestyle=':', zorder=4)
        all_vals = list(series)
        if pv is not None: all_vals.append(pv)
        if ev is not None: all_vals.append(ev)
        lo, hi = min(all_vals), max(all_vals)
        span = hi - lo if hi != lo else abs(lo) * 0.1 or 1.0
        ax.set_ylim(lo - 0.12 * span, hi + 0.12 * span)
        ax.set_title(f'{prefix}x_mean[{j}]  (mes. {j + 1})', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)
    for j in range(n_months, n_rows * n_cols):
        axes[j // n_cols, j % n_cols].set_visible(False)
    fig.suptitle(f'{prefix}X_mean – predikcia priemeru okna',
                 fontsize=10, y=1.01)
    fig.tight_layout()
    from IPython.display import display as _ipy_display
    _ipy_display(fig)
    plt.close(fig)


def plot_g_series(G_matrix, G_b_matrix, pred_g=None, pred_g_b=None,
                  emp_g=None, emp_g_b=None, lags=None, window_label=None,
                  X_mean_matrix=None, pred_X_mean=None, emp_X_mean=None,
                  style='analysis'):
    """
    Vizualizuje g-postupnosti (regulárnu aj neregulárnu časť).

    Pre každý komponent tri panely:
      - časový rad + predikcia + empirické g (ak je dostupné)
      - ACF  (max T-1 lagov)
      - PACF (max T-2 lagov, Levinson-Durbin)

    Parametre
    ---------
    G_matrix      : ndarray (T, r)       - regulárny g-rad
    G_b_matrix    : ndarray (T, r_b)     - neregulárny g_b-rad
    pred_g        : array-like (r,)/None - predikcia regulárneho radu
    pred_g_b      : array-like (r_b,)/None - predikcia neregulárneho radu
    emp_g         : array-like (r,)/None - empirické g pre nasledujúci krok
    emp_g_b       : array-like (r_b,)/None - empirické g_b pre nasledujúci krok
    lags          : int alebo None       - počet lagov pre ACF/PACF (None = max)
    window_label  : str alebo None       - popis okna pre nadpis grafu
    """
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    import matplotlib.pyplot as plt

    G   = np.asarray(G_matrix,   dtype=float)
    G_b = np.asarray(G_b_matrix, dtype=float)
    if G.ndim == 1:   G   = G.reshape(-1, 1)
    if G_b.ndim == 1: G_b = G_b.reshape(-1, 1)

    T = G.shape[0] if G.shape[0] > 0 else G_b.shape[0]
    # ACF: až T-1 lagov; PACF (Levinson-Durbin): až T-2 lagov
    max_acf_lags  = max(1, T - 1)
    max_pacf_lags = max(1, T // 2 - 1)  # statsmodels hard limit: <= 50% of sample
    acf_lags  = min(lags, max_acf_lags)  if lags is not None else max_acf_lags
    pacf_lags = min(lags, max_pacf_lags) if lags is not None else max_pacf_lags

    prefix = f'{window_label}  ' if window_label else ''
    _sy_g  = globals().get('start_year')
    _mh_g  = globals().get('m_hat', 0)
    _g_first_year = (_sy_g + _mh_g) if _sy_g is not None else None

    def _plot_one(series, pred_val, emp_val, title):
        if style == 'standard':
            fig, ax0 = plt.subplots(1, 1, figsize=(9, 4))
        else:
            fig, axes = plt.subplots(1, 3, figsize=(16, 4))
            ax0 = axes[0]

        # -- Časový rad + predikcia + empirické g
        xs = np.arange(len(series))
        ax0.plot(xs, series, marker='o', markersize=4,
                 linewidth=1.2, color='steelblue', label='G-seria')
        if pred_val is not None:
            ax0.plot([len(series) - 1, len(series)],
                     [series[-1], pred_val],
                     color='red', linewidth=1.2, linestyle='--')
            ax0.scatter([len(series)], [pred_val],
                        color='red', s=60, zorder=5,
                        marker='*', label='Predikcia')
        if emp_val is not None:
            ax0.scatter([len(series)], [emp_val],
                        color='green', s=70, zorder=6,
                        marker='D', label='Empiricke g')
            if pred_val is not None:
                ax0.plot([len(series), len(series)], [pred_val, emp_val],
                         color='grey', linewidth=1.0, linestyle=':', zorder=4)

        # Os Y: priblíženie na rozsah dát + malá marža, bez vzdialenej nuly
        all_vals = list(series)
        if pred_val is not None: all_vals.append(pred_val)
        if emp_val  is not None: all_vals.append(emp_val)
        lo, hi = min(all_vals), max(all_vals)
        span = hi - lo if hi != lo else abs(lo) * 0.1 or 1.0
        ax0.set_ylim(lo - 0.12 * span, hi + 0.12 * span)

        _ptitle = f'{prefix}{title}' if style == 'standard' else f'{prefix}{title}  --  casovy rad'
        ax0.set_title(_ptitle, fontsize=12)
        ax0.set_xlabel('Index okna', fontsize=11)
        ax0.legend(fontsize=10)
        ax0.tick_params(labelsize=10)
        ax0.grid(True, alpha=0.3)
        if style == 'standard':
            ax0.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        if _g_first_year is not None:
            _tks = list(range(len(series) + 1))
            ax0.set_xticks(_tks)
            ax0.set_xticklabels(
                [f"'{(_g_first_year + i) % 100:02d}" for i in _tks],
                fontsize=10)

        if style != 'standard':
            # -- ACF (max T-1 lagov)
            plot_acf(series, ax=axes[1], lags=acf_lags, alpha=0.05,
                     title=f'{prefix}{title}  --  ACF')
            axes[1].set_xlabel('Lag', fontsize=11)
            axes[1].tick_params(labelsize=10)

            # -- PACF (max T-2 lagov, Levinson-Durbin)
            plot_pacf(series, ax=axes[2], lags=pacf_lags, alpha=0.05,
                      method='ld', title=f'{prefix}{title}  --  PACF')
            axes[2].set_xlabel('Lag', fontsize=11)
            axes[2].tick_params(labelsize=10)

        fig.tight_layout()
        from IPython.display import display as _ipy_display
        _ipy_display(fig)
        plt.close(fig)

    r   = G.shape[1]   if G.size   > 0 else 0
    r_b = G_b.shape[1] if G_b.size > 0 else 0

    if r == 0 and r_b == 0:
        print('plot_g_series: obidve matice su prazdne.')
        return

    pred_g_arr   = np.asarray(pred_g,   dtype=float) if pred_g   is not None else None
    pred_g_b_arr = np.asarray(pred_g_b, dtype=float) if pred_g_b is not None else None
    emp_g_arr    = np.asarray(emp_g,    dtype=float) if emp_g    is not None else None
    emp_g_b_arr  = np.asarray(emp_g_b,  dtype=float) if emp_g_b  is not None else None

    for i in range(r):
        pv = float(pred_g_arr[i])  if (pred_g_arr  is not None and i < len(pred_g_arr))  else None
        ev = float(emp_g_arr[i])   if (emp_g_arr   is not None and i < len(emp_g_arr))   else None
        _plot_one(G[:, i], pv, ev, f'$g_{{{i+1}}}$')

    for i in range(r_b):
        pv = float(pred_g_b_arr[i]) if (pred_g_b_arr is not None and i < len(pred_g_b_arr)) else None
        ev = float(emp_g_b_arr[i])  if (emp_g_b_arr  is not None and i < len(emp_g_b_arr))  else None
        _plot_one(G_b[:, i], pv, ev, f'$g_{{b,{i+1}}}$')

    if X_mean_matrix is not None and pred_X_mean is not None:
        _plot_x_mean_series(np.asarray(X_mean_matrix, dtype=float),
                            pred_X_mean, emp_X_mean, prefix=prefix)


# -- Globálny sklad g-sérií: zoznam dictov, jeden záznam na predikčné okno --
_g_store_list = []  # [{G, G_b, pred_G, pred_G_b, emp_G, emp_G_b, label}, ...]


def clear_g_store():
    """Vymaže uložené g-série. Volaj pred novým predikčným cyklom."""
    global _g_store_list
    _g_store_list = []


def _plot_yearly_means_series(yearly_means, row_mean_pred, emp_row_mean=None, prefix='', style='analysis'):
    """
    3-panelový graf radu ročných priemerov (row centering):
      časový rad + ACF + PACF.
    Analógia plot_g_series pre 1D rad ročných priemerov.
    """
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    import matplotlib.pyplot as plt
    ym = np.asarray(yearly_means, dtype=float)
    if ym.size == 0:
        return
    T = len(ym)
    max_acf_lags  = max(1, T - 1)
    max_pacf_lags = max(1, T // 2 - 1)

    if style == 'standard':
        fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
    else:
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        ax = axes[0]

    # -- Časový rad + predikcia + empirický priemer --
    ax.plot(np.arange(T), ym, marker='o', markersize=4, linewidth=1.2,
            color='steelblue', label='Rocny priemer')
    pv = None
    if row_mean_pred is not None:
        pv = float(row_mean_pred)
        ax.plot([T - 1, T], [ym[-1], pv], color='red',
                linewidth=1.2, linestyle='--')
        ax.scatter([T], [pv], color='red', s=60, zorder=5,
                   marker='*', label='Predikcia (ARIMA)')
    if emp_row_mean is not None:
        ev = float(emp_row_mean)
        ax.scatter([T], [ev], color='green', s=55, zorder=6,
                   marker='D', label='Empiricky priemer')
        if pv is not None:
            ax.plot([T, T], [pv, ev], color='grey',
                    linewidth=0.8, linestyle=':', zorder=4)
    all_vals = list(ym)
    if pv is not None: all_vals.append(pv)
    if emp_row_mean is not None: all_vals.append(float(emp_row_mean))
    lo, hi = min(all_vals), max(all_vals)
    span = hi - lo if hi != lo else abs(lo) * 0.1 or 1.0
    ax.set_ylim(lo - 0.12 * span, hi + 0.12 * span)
    _ym_title = f'{prefix}Rocne priemery' if style == 'standard' else f'{prefix}Rocne priemery  --  casovy rad'
    ax.set_title(_ym_title, fontsize=12)
    ax.set_xlabel('Rok (index)', fontsize=11)
    ax.legend(fontsize=10)
    ax.tick_params(labelsize=10)
    ax.grid(True, alpha=0.3)
    if style == 'standard':
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    _sy_ym = globals().get('start_year')
    if _sy_ym is not None:
        _tks_ym = list(range(T + 1))
        ax.set_xticks(_tks_ym)
        ax.set_xticklabels(
            [f"'{(_sy_ym + i) % 100:02d}" for i in _tks_ym],
            fontsize=10)

    if style != 'standard':
        # -- ACF --
        plot_acf(ym, ax=axes[1], lags=max_acf_lags, alpha=0.05,
                 title=f'{prefix}Rocne priemery  --  ACF')
        axes[1].set_xlabel('Lag', fontsize=11)
        axes[1].tick_params(labelsize=10)

        # -- PACF --
        plot_pacf(ym, ax=axes[2], lags=max_pacf_lags, alpha=0.05,
                  method='ld', title=f'{prefix}Rocne priemery  --  PACF')
        axes[2].set_xlabel('Lag', fontsize=11)
        axes[2].tick_params(labelsize=10)

    fig.tight_layout()
    from IPython.display import display as _ipy_display
    _ipy_display(fig)
    plt.close(fig)



def _plot_g_overview(entries_chron, centering='none'):
    _sy  = globals().get('start_year')
    _mh  = globals().get('m_hat', 0)

    def _draw(key_G, key_pred, key_emp, comp_label):
        import matplotlib.pyplot as plt
        base = entries_chron[-1]
        G_base = base.get(key_G)
        if G_base is None:
            return
        G_base = __import__('numpy').asarray(G_base, dtype=float)
        if G_base.ndim == 1:
            G_base = G_base.reshape(-1, 1)
        if G_base.size == 0:
            return
        for ci in range(G_base.shape[1]):
            series = G_base[:, ci]
            T = len(series)
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            _last_emp_arr = entries_chron[-1].get(key_emp)
            _last_ev_ext = float(_last_emp_arr[ci]) if _last_emp_arr is not None and ci < len(_last_emp_arr) else None
            _ext_xs = list(range(T)) + ([T] if _last_ev_ext is not None else [])
            _ext_ys = list(series) + ([_last_ev_ext] if _last_ev_ext is not None else [])
            ax.plot(_ext_xs, _ext_ys, marker='o', markersize=3,
                    linewidth=1.2, color='steelblue', label='Tréningová hodnota')
            all_v = list(series)
            _lbl_p = 'Predikcia'
            _lbl_e = 'Empirická hodnota'
            for entry in entries_chron:
                G_e = entry.get(key_G)
                if G_e is None:
                    T_e = T
                else:
                    G_e = __import__('numpy').asarray(G_e, dtype=float)
                    if G_e.ndim == 1:
                        G_e = G_e.reshape(-1, 1)
                    T_e = G_e.shape[0]
                pred = entry.get(key_pred)
                emp  = entry.get(key_emp)
                pv = float(pred[ci]) if pred is not None and ci < len(pred) else None
                ev = float(emp[ci])  if emp  is not None and ci < len(emp)  else None
                if pv is not None:
                    ax.scatter([T_e], [pv], color='red', s=60, zorder=5,
                               marker='*', label=_lbl_p)
                    all_v.append(pv)
                    _lbl_p = '_nolegend_'
                if ev is not None:
                    ax.scatter([T_e], [ev], color='green', s=55, zorder=6,
                               marker='D', label=_lbl_e)
                    all_v.append(ev)
                    _lbl_e = '_nolegend_'
                if pv is not None and ev is not None:
                    ax.plot([T_e, T_e], [pv, ev], color='grey',
                            linewidth=0.8, linestyle=':', zorder=4)
            lo, hi = min(all_v), max(all_v)
            span = hi - lo if hi != lo else abs(lo) * 0.1 or 1.0
            ax.set_ylim(lo - 0.12 * span, hi + 0.12 * span)
            _latex_t = f'$g_{{{ci+1}}}$' if comp_label == 'g_regular' else f'$g_{{b,{ci+1}}}$'
            ax.set_title(_latex_t, fontsize=14)
            ax.set_xlabel('Rok', fontsize=17)
            ax.legend(fontsize=11, loc='lower right')
            ax.grid(True, alpha=0.3)
            if _sy is not None:
                _first = _sy + _mh
                _tks = list(range(T + 1))
                ax.set_xticks(_tks)
                ax.set_xticklabels([f"'{(_first + i) % 100:02d}" for i in _tks],
                                   fontsize=11)
            fig.tight_layout()
            from IPython.display import display as _ipy_display
            _ipy_display(fig)
            plt.close(fig)

    _draw('G',   'pred_G',   'emp_G',   'g_regular')
    _draw('G_b', 'pred_G_b', 'emp_G_b', 'g_nonregular')

    if centering != 'row':
        return
    import matplotlib.pyplot as plt
    import numpy as np
    base = entries_chron[-1]
    base_ym = base.get('yearly_means')
    if base_ym is None:
        return
    ym = np.asarray(base_ym, dtype=float)
    T_ym = len(ym)
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    _last_emp_ym = entries_chron[-1].get('emp_row_mean')
    _last_ev_ym = float(_last_emp_ym) if _last_emp_ym is not None else None
    _ext_xs_ym = list(range(T_ym)) + ([T_ym] if _last_ev_ym is not None else [])
    _ext_ys_ym = list(ym) + ([_last_ev_ym] if _last_ev_ym is not None else [])
    ax.plot(_ext_xs_ym, _ext_ys_ym, marker='o', markersize=3,
            linewidth=1.2, color='steelblue', label='Tréningová hodnota')
    all_v = list(ym)
    _lbl_p = 'Predikcia'
    _lbl_e = 'Empirický priemer'
    for entry in entries_chron:
        ym_e = entry.get('yearly_means')
        if ym_e is None:
            continue
        T_e = len(ym_e)
        pv = float(entry['row_mean_pred']) if entry.get('row_mean_pred') is not None else None
        ev = float(entry['emp_row_mean'])  if entry.get('emp_row_mean')  is not None else None
        if pv is not None:
            ax.scatter([T_e], [pv], color='red', s=60, zorder=5,
                       marker='*', label=_lbl_p)
            all_v.append(pv)
            _lbl_p = '_nolegend_'
        if ev is not None:
            ax.scatter([T_e], [ev], color='green', s=55, zorder=6,
                       marker='D', label=_lbl_e)
            all_v.append(ev)
            _lbl_e = '_nolegend_'
        if pv is not None and ev is not None:
            ax.plot([T_e, T_e], [pv, ev], color='grey',
                    linewidth=0.8, linestyle=':', zorder=4)
    lo, hi = min(all_v), max(all_v)
    span = hi - lo if hi != lo else abs(lo) * 0.1 or 1.0
    ax.set_ylim(lo - 0.12 * span, hi + 0.12 * span)
    ax.set_title('Ročné priemery', fontsize=14)
    ax.set_xlabel('Rok', fontsize=17)
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.3)
    if _sy is not None:
        _tks = list(range(T_ym + 1))
        ax.set_xticks(_tks)
        ax.set_xticklabels([f"'{(_sy + i) % 100:02d}" for i in _tks], fontsize=11)
    fig.tight_layout()
    from IPython.display import display as _ipy_display
    _ipy_display(fig)
    plt.close(fig)


def plot_stored_g_series(lags=None, centering='column', style='analysis'):
    """
    Vykreslí g-rady pre všetky walk-forward okná uložené počas posledného
    predikčného cyklu (najstaršie okno prvé = chronologicky).

    centering : 'none' | 'column' | 'row'
        'column' - zobrazuje aj 12-panelový grid mesačných priemerov.
        'row'    - zobrazuje 1D rad ročných priemerov s ARIMA predikciou.
        'none'   - zobrazuje iba g-série bez centrovacích grafov.
    """
    if not _g_store_list:
        print('Ziadne g-serie nie su ulozene. Najprv spusti predikciu.')
        return
    if style == 'standard':
        _plot_g_overview(list(reversed(_g_store_list)), centering=centering)
        return
    for entry in reversed(_g_store_list):
        # X_mean grid: iba pre column centering
        x_mean_matrix = entry.get('X_mean_matrix') if centering == 'column' else None
        pred_x_mean   = entry.get('pred_X_mean')   if centering == 'column' else None
        emp_x_mean    = entry.get('emp_X_mean')    if centering == 'column' else None
        plot_g_series(
            entry['G'], entry['G_b'],
            pred_g=entry.get('pred_G'),
            pred_g_b=entry.get('pred_G_b'),
            emp_g=entry.get('emp_G'),
            emp_g_b=entry.get('emp_G_b'),
            lags=lags,
            window_label=entry.get('label'),
            X_mean_matrix=x_mean_matrix,
            pred_X_mean=pred_x_mean,
            emp_X_mean=emp_x_mean,
            style=style,
        )
        # Row centering: zobraz 1D rad ročných priemerov
        if centering == 'row' and entry.get('yearly_means') is not None:
            prefix = f"{entry.get('label')}  " if entry.get('label') else ''
            _plot_yearly_means_series(
                entry['yearly_means'],
                entry.get('row_mean_pred'),
                emp_row_mean=entry.get('emp_row_mean'),
                prefix=prefix,
                style=style,
            )


def plot_stored_decomposition(export_vector, m_total, n=12, centering='column'):
    """
    Pre každé uložené walk-forward okno vykreslí rozklad predikcie na:
      - priemer okna / ročná úrove^ (centrovanie) - modrá / fialová
      - regulárna časť a_l                        - zelená
      - neregulárna časť A_N_pred                  - oranžová
    s prekreslením skutočných hodnôt (červená) a celkovej predikcie (čierna).

    centering : 'none' | 'column' | 'row'
        'none'   - xm bar sa nezobrazuje (nulový príspevok).
        'column' - xm bar = mesačný priemer okna (modrá).
        'row'    - xm bar = ročná úrove^ (fialová, rovnaká pre všetky mesiace).
    """
    import matplotlib.pyplot as plt
    if not _g_store_list:
        print('Ziadne predikcie nie su ulozene. Spusti prediction loop.')
        return
    for entry in reversed(_g_store_list):
        a_l   = entry.get('decomp_a_l')
        A_N   = entry.get('decomp_A_N')
        xm    = entry.get('decomp_X_mean')
        m_val = entry.get('decomp_m')
        label = entry.get('label', '')
        if a_l is None or m_val is None:
            continue
        a_l = np.asarray(a_l, dtype=float)
        A_N = np.asarray(A_N, dtype=float)
        xm  = np.asarray(xm,  dtype=float) if xm is not None else np.zeros(n)
        total  = xm + a_l + A_N
        months = np.arange(1, n + 1)

        # Skutočné dáta pre predikovanú periódu
        s, e = m_val * n, (m_val + 1) * n
        ev = np.asarray(export_vector, dtype=float)
        actual = ev[s:e] if e <= len(ev) else None

        fig, ax = plt.subplots(figsize=(13, 5))

        # -- Skladané stĺpce (rozdelenie pos/neg pre správne vykreslenie) --
        # Stĺpec centrovania: farba/label závisí od módu; skrytý pre 'none'
        if centering == 'column':
            ax.bar(months, xm, width=0.65, color='cornflowerblue', alpha=0.80,
                   label=r'$\bar{x}$ – priemer okna (centrovanie)')
        elif centering == 'row':
            ax.bar(months, xm, width=0.65, color='mediumpurple', alpha=0.80,
                   label=r'$\bar{x}$ – rocna uroven (centrovanie)')
        # 'none': bez stĺpca (xm sú nuly)

        al_pos = np.where(a_l >= 0, a_l, 0.0)
        al_neg = np.where(a_l <  0, a_l, 0.0)
        ax.bar(months, al_pos, bottom=xm,       width=0.65,
               color='mediumseagreen', alpha=0.85,
               label='$a_l$ – regulárna časť')
        ax.bar(months, al_neg, bottom=xm,       width=0.65,
               color='mediumseagreen', alpha=0.85)

        base_an = xm + a_l
        an_pos  = np.where(A_N >= 0, A_N, 0.0)
        an_neg  = np.where(A_N <  0, A_N, 0.0)
        ax.bar(months, an_pos, bottom=base_an,  width=0.65,
               color='darkorange', alpha=0.85,
               label='$A_N$ – neregulárna časť')
        ax.bar(months, an_neg, bottom=base_an,  width=0.65,
               color='darkorange', alpha=0.85)

        # -- Prekrývajúce čiary --
        ax.plot(months, total, 'k--', linewidth=1.6, zorder=5,
                label='Predikcia (súčet)')
        if actual is not None:
            ax.plot(months, actual, 'r-o', linewidth=2.0, markersize=5,
                    zorder=6, label='Skutočnosť')

        ax.set_xlabel('Mesiac', fontsize=15)
        ax.set_ylabel('Hodnota', fontsize=14)
        ax.set_title(f'Rozklad predikcie  [{label}]',
                     fontsize=14, fontweight='bold')
        ax.set_xticks(months)
        ax.tick_params(labelsize=13)
        ax.legend(fontsize=11, framealpha=0.8, loc='upper left')
        ax.grid(True, axis='y', alpha=0.22, linestyle='--')
        ax.spines[['top', 'right']].set_visible(False)
        fig.tight_layout()
        from IPython.display import display as _ipy_display
        _ipy_display(fig)
        plt.close(fig)


def plot_combined_decomposition(export_vector, m_total, n=12, centering='column'):
    """
    Jeden spojený graf všetkých walk-forward predikcií QS-PM –
    mriežka podgrafov (1 na okno), každý zobrazuje rozklad na
    centrovanie / regulárnu / neregulárnu časť + skutočnosť.
    """
    import matplotlib.pyplot as plt
    import math
    if not _g_store_list:
        print('Ziadne predikcie nie su ulozene. Spusti prediction loop.')
        return

    entries = list(reversed(_g_store_list))   # chronologicky: najstarsia 1.
    p_wins  = len(entries)
    ncols   = min(p_wins, 4)
    nrows   = math.ceil(p_wins / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 6.5, nrows * 5.2),
                              sharey=False)
    axes = np.array(axes).flatten() if p_wins > 1 else [axes]

    months = np.arange(1, n + 1)
    ev     = np.asarray(export_vector, dtype=float)

    _legend_handles = []

    for ax_i, (ax, entry) in enumerate(zip(axes, entries)):
        a_l   = np.asarray(entry.get('decomp_a_l',  [0]*n), dtype=float)
        A_N   = np.asarray(entry.get('decomp_A_N',  [0]*n), dtype=float)
        xm    = np.asarray(entry.get('decomp_X_mean', [0]*n), dtype=float)
        m_val = entry.get('decomp_m')
        label = entry.get('label', '')
        total = xm + a_l + A_N

        # Centrovanie
        if centering == 'column':
            b0 = ax.bar(months, xm, width=0.65, color='cornflowerblue',
                        alpha=0.80, label=r'$\bar{x}$ – centrovanie')
        elif centering == 'row':
            b0 = ax.bar(months, xm, width=0.65, color='mediumpurple',
                        alpha=0.80, label=r'$\bar{x}$ – centrovanie')
        else:
            b0 = None

        al_pos = np.where(a_l >= 0, a_l, 0.0)
        al_neg = np.where(a_l <  0, a_l, 0.0)
        b1 = ax.bar(months, al_pos, bottom=xm, width=0.65,
                    color='mediumseagreen', alpha=0.85,
                    label='$a_l$ – regulárna časť')
        ax.bar(months, al_neg, bottom=xm, width=0.65,
               color='mediumseagreen', alpha=0.85)

        base_an = xm + a_l
        an_pos  = np.where(A_N >= 0, A_N, 0.0)
        an_neg  = np.where(A_N <  0, A_N, 0.0)
        b2 = ax.bar(months, an_pos, bottom=base_an, width=0.65,
                    color='darkorange', alpha=0.85,
                    label='$A_N$ – neregulárna časť')
        ax.bar(months, an_neg, bottom=base_an, width=0.65,
               color='darkorange', alpha=0.85)

        l1, = ax.plot(months, total, 'k--', linewidth=1.5, zorder=5,
                      label='Predikcia')
        s_e, e_e = m_val * n, (m_val + 1) * n
        actual = ev[s_e:e_e] if m_val is not None and e_e <= len(ev) else None
        if actual is not None:
            l2, = ax.plot(months, actual, 'r-o', linewidth=2.0,
                          markersize=4, zorder=6, label='Skutočnosť')

        # Zachovaj handly pre legendu (len z prvého podgrafu)
        if ax_i == 0:
            _legend_handles = [h for h in
                [b0, b1, b2, l1, (l2 if actual is not None else None)]
                if h is not None]

        ax.set_title(label, fontsize=25, fontweight='bold')
        ax.set_xticks(months)
        ax.tick_params(labelsize=20)
        ax.set_xlabel('Mesiac', fontsize=25)
        ax.grid(True, axis='y', alpha=0.20, linestyle='--')
        ax.spines[['top', 'right']].set_visible(False)

    # Skry prázdne podgrafy
    for ax in axes[p_wins:]:
        ax.set_visible(False)

    fig.legend(handles=_legend_handles, loc='lower center',
               ncol=len(_legend_handles), fontsize=25,
               framealpha=0.85, bbox_to_anchor=(0.5, 0.0),
               handlelength=3.0, handleheight=1.8,
               handletextpad=1.0, columnspacing=2.5,
               borderpad=1.2, markerscale=1.5)
    fig.tight_layout(rect=[0, 0.14, 1, 1])
    from IPython.display import display as _ipy_display
    _ipy_display(fig)
    plt.close(fig)


# -- Register prediktorov --
# Protokol: forecaster(series: np.ndarray) -> float
# Nové prediktory pridaj tu; referencuj kľúčom typu string pri spustení.

def make_arima_forecaster(order):
    """Factory: ARIMA prediktor s pevným rádom (p,d,q)."""
    def _f(series):
        return float(ARIMA(series, order=order).fit().forecast(steps=1)[0])
    _f.__name__ = f'arima{order}'
    return _f


def make_linear_forecaster():
    """Factory: lineárna extrapolácia trendu y = a*t + b, predikcia v t = T."""
    def _f(series):
        t = np.arange(len(series), dtype=float)
        a, b = np.polyfit(t, series, 1)
        return float(a * len(series) + b)
    _f.__name__ = 'linear'
    return _f



def make_auto_arima_forecaster(log_orders=False, **kwargs):
    """Factory: auto_arima prediktor (pmdarima) – automatický výber rádu ARIMA."""
    defaults = dict(seasonal=False, d=1, error_action='ignore', suppress_warnings=True)
    defaults.update(kwargs)
    def _f(series):
        model = auto_arima(series, **defaults)
        if log_orders:
            print(f'  [auto_arima] chosen order={model.order}', flush=True)
        return float(model.predict(1)[0])
    _f.__name__ = 'auto_arima'
    return _f


def make_anchored_linear_forecaster(k=1):
    """
    Factory: regresia cez fixny bod.

    Kotva = median poslednych k hodnot (pri k=1 je to posledna hodnota).
    Sklon je odhadnuty OLS bez interceptu na posunutych premennych:

        xs = t - (T-1),   ys = y_t - anchor_y
        beta = sum(xs * ys) / sum(xs^2)
        forecast = anchor_y + beta

    Ekvivalentne: random-walk-with-drift s driftom odhadnutym z celeho
    historickeho okna, kotvenym v poslednom (medianovom) bode.
    """
    def _f(series):
        T = len(series)
        if T < 2:
            return float(series[-1])
        anchor_y = float(np.median(series[-k:]))
        xs = np.arange(T, dtype=float) - (T - 1)   # anchor at x=0
        ys = series - anchor_y
        denom = float(np.dot(xs, xs))
        beta  = float(np.dot(xs, ys) / denom) if denom != 0.0 else 0.0
        return anchor_y + beta          # one step ahead: x = +1
    _f.__name__ = f'anchored_linear_k{k}'
    return _f

def make_wls_forecaster(lam=0.85):
    """WLS s exponencialnym vahovanim: novsi bod = vacsia vaha (lambda^(T-1-t))."""
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        t = np.arange(T, dtype=float)
        w = lam ** (T - 1 - t)
        A = np.column_stack([t, np.ones(T)])
        AtW = A.T * w
        try:
            params = np.linalg.solve(AtW @ A, AtW @ series)
        except np.linalg.LinAlgError:
            params = np.polyfit(t, series, 1)[::-1]
        return float(params[0] * T + params[1])
    _f.__name__ = f'wls_lam{lam}'
    return _f

def make_anchored_wls_forecaster(k=1, lam=0.85):
    """Regresia cez posledny bod s exponencialnym vahovanim sklonu."""
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        anchor_y = float(np.median(series[-k:]))
        t = np.arange(T, dtype=float)
        w = lam ** (T - 1 - t)
        xs = t - (T - 1)
        ys = series - anchor_y
        denom = float(np.dot(w, xs ** 2))
        beta  = float(np.dot(w * xs, ys) / denom) if denom != 0.0 else 0.0
        return anchor_y + beta
    _f.__name__ = f'anchored_wls_k{k}_lam{lam}'
    return _f

def make_local_linear_forecaster(k=4):
    """OLS iba na poslednych k bodoch; adaptuje sa na nedavny trend."""
    def _f(series):
        k_eff = min(k, len(series))
        seg = series[-k_eff:]
        T_full = len(series)
        t = np.arange(T_full - k_eff, T_full, dtype=float)
        if k_eff < 2: return float(seg[-1])
        a, b = np.polyfit(t, seg, 1)
        return float(a * T_full + b)
    _f.__name__ = f'local_linear_k{k}'
    return _f

def make_theil_sen_forecaster():
    """Median vsetkych pairwise sklonov; breakdown point 29%."""
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        t = np.arange(T, dtype=float)
        slopes = [(series[j] - series[i]) / (t[j] - t[i])
                  for i in range(T) for j in range(i + 1, T)]
        beta = float(np.median(slopes))
        intercept = float(np.median(series - beta * t))
        return beta * T + intercept
    _f.__name__ = 'theil_sen'
    return _f

def make_repeated_median_forecaster():
    """Median sklonov pre kazdy bod; breakdown point 50%."""
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        t = np.arange(T, dtype=float)
        medians_i = []
        for i in range(T):
            pw = [(series[j] - series[i]) / (t[j] - t[i]) for j in range(T) if j != i]
            medians_i.append(float(np.median(pw)))
        beta = float(np.median(medians_i))
        intercept = float(np.median(series - beta * t))
        return beta * T + intercept
    _f.__name__ = 'repeated_median'
    return _f

def make_holt_forecaster(alpha=0.8, beta=0.3):
    """Holtovo exponencialne vyhladenie: uroven + trend (dva param. alpha, beta)."""
    def _f(series):
        if len(series) < 2: return float(series[-1])
        l = float(series[0])
        b = float(series[1] - series[0])
        for y in series[1:]:
            l_p, b_p = l, b
            l = alpha * float(y) + (1 - alpha) * (l_p + b_p)
            b = beta  * (l - l_p) + (1 - beta)  * b_p
        return l + b
    _f.__name__ = f'holt_a{alpha}_b{beta}'
    return _f

def make_double_ema_forecaster(alpha=0.5):
    """Brownova dvojita EMA; jeden parameter alpha."""
    def _f(series):
        if len(series) < 2: return float(series[-1])
        e1 = float(series[0])
        for y in series[1:]:
            e1 = alpha * float(y) + (1 - alpha) * e1
        e2 = float(series[0])
        tmp = float(series[0])
        for y in series[1:]:
            tmp = alpha * float(y) + (1 - alpha) * tmp
            e2  = alpha * tmp       + (1 - alpha) * e2
        a_br = 2 * e1 - e2
        b_br = (alpha / (1 - alpha)) * (e1 - e2)
        return a_br + b_br
    _f.__name__ = f'double_ema_a{alpha}'
    return _f

def make_drift_mean_forecaster():
    """Posledna hodnota + priemer prvych diferencii (= ARIMA(0,1,0)+drift)."""
    def _f(series):
        if len(series) < 2: return float(series[-1])
        return float(series[-1]) + float(np.mean(np.diff(series.astype(float))))
    _f.__name__ = 'drift_mean'
    return _f

def make_drift_median_forecaster():
    """Posledna hodnota + median prvych diferencii; robustna verzia drift_mean."""
    def _f(series):
        if len(series) < 2: return float(series[-1])
        return float(series[-1]) + float(np.median(np.diff(series.astype(float))))
    _f.__name__ = 'drift_median'
    return _f

def make_drift_ewm_forecaster(lam=0.85):
    """Posledna hodnota + exponencialne vahovany priemer diferencii."""
    def _f(series):
        if len(series) < 2: return float(series[-1])
        diffs = np.diff(series.astype(float))
        D = len(diffs)
        w = np.array([lam ** (D - 1 - i) for i in range(D)])
        w /= w.sum()
        return float(series[-1]) + float(np.dot(w, diffs))
    _f.__name__ = f'drift_ewm_lam{lam}'
    return _f

def make_ridge_forecaster(alpha=1.0):
    """OLS s L2 penaltou na sklon; zmrsuje agresivnu extrapolaciu smerom k nule."""
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        t = np.arange(T, dtype=float)
        t_c = t - t.mean()
        X = np.column_stack([t_c, np.ones(T)])
        pen = np.diag([alpha, 0.0])
        try:
            params = np.linalg.solve(X.T @ X + pen, X.T @ series)
        except np.linalg.LinAlgError:
            params = np.array([0.0, float(series[-1])])
        return float(params[0] * (T - t.mean()) + params[1])
    _f.__name__ = f'ridge_a{alpha}'
    return _f

def make_bayesian_linear_forecaster(prior_slope_std=1.0):
    """
    Bayesovska linearna regresia: prior beta ~ N(0, prior_slope_std^2),
    plochý prior na intercept, rozptyl sumu odhadnuty z OLS reziduii.
    Posterior mean = ridge s lambda = sigma^2 / prior_slope_std^2.
    """
    def _f(series):
        T = len(series)
        if T < 2: return float(series[-1])
        t = np.arange(T, dtype=float)
        t_c = t - t.mean()
        X = np.column_stack([t_c, np.ones(T)])
        try:
            p_ols = np.linalg.lstsq(X, series, rcond=None)[0]
            sig2  = max(float(np.mean((series - X @ p_ols) ** 2)), 1e-10)
        except Exception:
            sig2 = 1.0
        lam = sig2 / max(prior_slope_std ** 2, 1e-10)
        pen = np.diag([lam, 0.0])
        try:
            params = np.linalg.solve(X.T @ X + pen, X.T @ series)
        except np.linalg.LinAlgError:
            params = np.array([0.0, float(series[-1])])
        return float(params[0] * (T - t.mean()) + params[1])
    _f.__name__ = f'bayesian_linear_ps{prior_slope_std}'
    return _f

G_FORECASTERS = {
    'arima':               None,  # sentinel: pouzij ARIMA s explicitnymi radmi
    'linear':              make_linear_forecaster(),
    'linear_wls':          make_wls_forecaster(lam=0.85),
    'linear_anchored':     make_anchored_linear_forecaster(k=1),
    'linear_anchored_wls': make_anchored_wls_forecaster(k=1, lam=0.85),
    'auto_arima':          make_auto_arima_forecaster(),
    'auto_arima_log':      make_auto_arima_forecaster(log_orders=True),
}


def _resolve_forecaster(fc):
    """Preloží špecifikáciu prediktora na callable.

    Prijíma:
      'arima'               -> ARIMA s explicitnymi radmi (g_orders / g_b_orders)
      'linear'              -> unconstrained OLS
      'linear_wls'          -> WLS s exponencialnym vahovanim
      'linear_anchored'     -> regresia cez posledny bod
      'linear_anchored_wls' -> anchored + exponencialne vahy
      'auto_arima'          -> pmdarima.auto_arima
      'auto_arima_alt'      -> statsmodels AIC grid search (ARIMA)
      'auto_arima_sf'       -> statsforecast AutoARIMA (Nixtla, Hyndman-Khandakar)
      callable              -> series -> float
    """
    if callable(fc):
        return fc
    key = 'arima' if fc is None else fc
    if key not in G_FORECASTERS:
        raise KeyError(f'Neznamy forecaster {key!r}. Dostupne: {list(G_FORECASTERS)}')
    return G_FORECASTERS[key]  # 'arima' -> None -> predict_g_vectors pouzije g_orders


# -- predict_g_vectors --

def predict_g_vectors(G_matrix, G_b_matrix,
                      g_orders=None, g_b_orders=None,
                      g_forecaster=None, g_b_forecaster=None):
    """
    Predikuje nasledujúce hodnoty g a g_b pre každý komponent.

    Výber prediktora (aplikuje sa rovnomerne na všetky komponenty každej časti):
      g_forecaster / g_b_forecaster:
        None     -> ARIMA s g_orders / g_b_orders (pôvodné správanie)
        str      -> kľúč v G_FORECASTERS
        callable -> series -> float

    Ak je prediktor zadaný, g_orders / g_b_orders sa pre danú časť ignorujú.
    """
    G_matrix   = np.asarray(G_matrix,   dtype='float64')
    G_b_matrix = np.asarray(G_b_matrix, dtype='float64')
    if G_matrix.ndim == 1:   G_matrix   = G_matrix.reshape(-1, 1)
    if G_b_matrix.ndim == 1: G_b_matrix = G_b_matrix.reshape(-1, 1)
    r   = G_matrix.shape[1]
    r_b = G_b_matrix.shape[1]

    fc_g   = _resolve_forecaster(g_forecaster)
    fc_g_b = _resolve_forecaster(g_b_forecaster)

    predicted_g = []
    for i in range(r):
        series = G_matrix[:, i]
        if fc_g is not None:
            predicted_g.append([fc_g(series)])
        else:
            order = (g_orders[min(i, len(g_orders) - 1)] if (g_orders is not None and len(g_orders) > 0)
                     else ((1, 1, 4) if i == 0 else (0, 0, 3)))
            predicted_g.append(ARIMA(series, order=order).fit().forecast(steps=1))

    predicted_g_b = []
    for i in range(r_b):
        series = G_b_matrix[:, i]
        if fc_g_b is not None:
            predicted_g_b.append([fc_g_b(series)])
        else:
            order = (g_b_orders[i] if (g_b_orders is not None and i < len(g_b_orders))
                 else (1, 0, 0))
            predicted_g_b.append(ARIMA(series, order=order).fit().forecast(steps=1))

    predicted_g   = np.array(predicted_g,   dtype='float64').T if r   > 0 else np.zeros((1, 0))
    predicted_g_b = np.array(predicted_g_b, dtype='float64').T if r_b > 0 else np.zeros((1, 0))
    return predicted_g, predicted_g_b


def _get_indices(classifications):
    """Vráti zoradené zoznamy indexov komponentov r_n, r_b a redundant."""
    r_n = sorted(k for k, v in classifications.items()
                 if isinstance(k, int) and v == 'r_n')
    r_b = sorted(k for k, v in classifications.items()
                 if isinstance(k, int) and v == 'r_b')
    redundant = sorted(k for k, v in classifications.items()
                       if isinstance(k, int) and v == 'redundant')
    return r_n, r_b, redundant


def compute_g_classified(svd_results, r_n_indices):
    """Hodnoty g-série pre vybrané indexy regulárnych komponentov."""
    U, s = svd_results['U'], svd_results['s']
    return [float(U[-1, i] * s[i]) for i in r_n_indices]


def compute_g_b_classified(svd_results, r_b_indices, epsilon=1e-10,
                           transform='log', power_p=0.5, r_b=None):
    """
    Ako compute_g_b, ale používa klasifikované r_b_indices na zostavenie A_N.

    r_b_indices - ktoré SVD komponenty vstupujú do A_N
    r_b         - počet singulárnych hodnôt SVD matice B na predikciu.
                  None = len(r_b_indices).

    Vráti g_b (r_b hodnôt), Vt z B-SVD a c.
    """
    n_cols = svd_results['Vt'].shape[1]
    U, s, Vt = svd_results['U'], svd_results['s'], svd_results['Vt']

    if not r_b_indices:
        return np.array([]), np.zeros((0, n_cols)), 0.0
    an_indices = list(r_b_indices)

    n_out = r_b if r_b is not None else len(r_b_indices)

    if not an_indices:
        return np.array([]), np.zeros((0, n_cols)), 0.0

    A_N = sum(s[i] * U[:, i:i+1] @ Vt[i:i+1, :] for i in an_indices)
    min_AN = np.min(A_N)
    c = (-min_AN + epsilon) if min_AN <= 0 else epsilon
    f_tr = _make_transform(transform, power_p, epsilon)
    B = f_tr(A_N, c)

    res = process_vector_to_svd(B.flatten())
    g_b = [res['U'][-1, j] * res['s'][j] for j in range(n_out)]
    return np.array(g_b), res['Vt'][:n_out, :], c


def create_G_matrices_classified(window_size, m, classifications, time_series, period_length,
                                  transform='log', power_p=0.5,
                                  r_b=None, center='none', full_series=None):
    """
    Ako create_G_matrices, ale používa dict klasifikácií na výber komponentov
    pre regulárnu (g_n) a log-priestorovú (g_b) predikciu.

    center : 'none' | 'column' | 'row' | 'double'
        Režim centrovania aplikovaný na každé dátové okno pred SVD.
        'column' odčíta mesačné priemery (axis=0), tvar (12,).
        'row'    odčíta ročné priemery (axis=1), tvar (m_hat,).
        X_mean_list vždy ukladá tvar (period_length,); nuly pre iné módy.
    r_b : int alebo None
        Počet singulárnych hodnôt SVD matice B na predikciu. None = len(r_b_indices).
    """
    r_n_indices, r_b_indices, redundant_indices = _get_indices(classifications)
    G_matrix, G_b_matrix, c_list, X_mean_list = [], [], [], []
    svd_results = None
    Vt_b = np.zeros((0, period_length))

    for k in range(window_size, m):
        window = create_data_window_vector(time_series, k, period_length, window_size)
        svd_results = process_vector_to_svd(window, center=center)
        G_matrix.append(compute_g_classified(svd_results, r_n_indices))
        g_b, Vt_b, c = compute_g_b_classified(svd_results, r_b_indices,
                                              transform=transform, power_p=power_p,
                                              r_b=r_b)
        G_b_matrix.append(g_b)
        c_list.append(c)
        # X_mean_list: zmysluplné len pre 'column'; pre iné módy ukladaj nuly
        # (tvar zachovaný na period_length pre konzistentnosť)
        if center in ('column', 'double'):
            X_mean_list.append(svd_results['X_mean'].copy())
        else:
            X_mean_list.append(np.zeros(period_length))

    Vt_rn = (svd_results['Vt'][r_n_indices, :]
             if r_n_indices
             else np.zeros((0, period_length)))

    emp_G_val = emp_G_b_val = emp_X_mean_val = None
    _emp_series = full_series if full_series is not None else time_series
    try:
        window_next = create_data_window_vector(_emp_series, m, period_length, window_size)
        svd_next = process_vector_to_svd(window_next, center=center)
        emp_G_val   = np.array(compute_g_classified(svd_next, r_n_indices), dtype=float)
        emp_g_b_vals, _, _ = compute_g_b_classified(svd_next, r_b_indices,
                                                   transform=transform, power_p=power_p,
                                                   r_b=r_b)
        emp_G_b_val = np.array(emp_g_b_vals, dtype=float)
        emp_X_mean_val = svd_next['X_mean'].copy() if center in ('column', 'double') else None
    except (ValueError, IndexError):
        pass

    X_mean_matrix = np.array(X_mean_list) if X_mean_list else np.zeros((0, period_length))
    _grand_mean_last = float(svd_results['grand_mean']) if svd_results is not None and center == 'double' else 0.0
    _g_store_list.append({
        'G': np.array(G_matrix), 'G_b': np.array(G_b_matrix),
        'emp_G': emp_G_val, 'emp_G_b': emp_G_b_val,
        'pred_G': None, 'pred_G_b': None, 'label': None,
        'X_mean_matrix': X_mean_matrix, 'emp_X_mean': emp_X_mean_val, 'pred_X_mean': None,
        'grand_mean': _grand_mean_last,
    })

    X_mean_last = (svd_results['X_mean'].copy()
                   if svd_results is not None and center in ('column', 'double')
                   else np.zeros(period_length))
    return np.array(G_matrix), np.array(G_b_matrix), Vt_rn, Vt_b, c_list, X_mean_last, X_mean_matrix


def one_period_prediction_classified(m_hat, m, classifications, n, export_vector,
                                     g_orders=None, g_b_orders=None,
                                     g_forecaster=None, g_b_forecaster=None,
                                     transform='log', power_p=0.5,
                                     r_b=None, center='none',
                                     column_mean_order=(1, 1, 0),
                                     row_mean_order=(1, 1, 0),
                                     full_series=None):
    """
    Náhrada za one_period_prediction, používa klasifikácie komponentov
    z classify_eigenvalues_interactive.

    center : 'none' | 'column' | 'row' | 'double'
        'none'   - bez centrovania; SVD pracuje s raw maticou.
        'column' - odčíta mesačné priemery okna; priemery predikované ARIMA
                   s rádom column_mean_order a pridané späť k rekonštrukcii.
        'row'    - odčíta ročné priemery okna; ďalší ročný priemer predikovaný
                   ARIMA s rádom row_mean_order a pridaný späť ako skalár.
        'double' - DSVD (Zhang): odčíta mesačné aj ročné priemery + grand mean;
                   rekonštrukcia = SVD + col_mean_pred + row_mean_pred - grand_mean.

    column_mean_order : (p, d, q)
        Rád ARIMA na predikciu mesačných priemerov ďalšieho okna (12 sérií).
    row_mean_order : (p, d, q)
        Rád ARIMA na predikciu ročnej úrovne ďalšieho roka (1 séria).
    """
    r_n_indices, r_b_indices, redundant_indices = _get_indices(classifications)
    r   = len(r_n_indices)
    if r_b is not None:
        n_b_svd = r_b
    else:
        n_b_svd = len(r_b_indices)

    G_matrix, G_b_matrix, Vt_rn, Vt_b, c_list, X_mean_last, X_mean_matrix = create_G_matrices_classified(
        window_size=m_hat, m=m, classifications=classifications,
        time_series=export_vector, period_length=n,
        transform=transform, power_p=power_p,
        r_b=r_b, center=center,
        full_series=full_series,
    )
    predicted_g, predicted_g_b = predict_g_vectors(
        G_matrix, G_b_matrix,
        g_orders=g_orders, g_b_orders=g_b_orders,
        g_forecaster=g_forecaster, g_b_forecaster=g_b_forecaster,
    )

    # -- Predikcia centrovacieho člena pre predikčnú periódu --
    X_mean_pred  = np.zeros(n)
    row_mean_pred = 0.0
    yearly_means  = np.array([])   # populated when center == 'row'
    emp_row_mean  = None           # empirical yearly mean of forecast period
    if center in ('column', 'double') and X_mean_matrix.shape[0] >= 3:
        try:
            X_mean_pred = np.array([
                float(ARIMA(X_mean_matrix[:, j], order=column_mean_order).fit().forecast(steps=1)[0])
                for j in range(X_mean_matrix.shape[1])
            ])
        except Exception:
            X_mean_pred = X_mean_last.copy()
    elif center in ('column', 'double'):
        X_mean_pred = X_mean_last.copy()
    if center in ('row', 'double'):
        yearly_means = np.array([
            float(np.mean(export_vector[k * n : (k + 1) * n]))
            for k in range(m)
        ])
        try:
            row_mean_pred = float(
                ARIMA(yearly_means, order=row_mean_order).fit().forecast(steps=1)[0]
            )
        except Exception:
            row_mean_pred = float(yearly_means[-1])
        _s, _e = m * n, (m + 1) * n
        _ev_full = np.asarray(full_series if full_series is not None else export_vector)
        if _e <= len(_ev_full):
            emp_row_mean = float(np.mean(_ev_full[_s:_e]))

    if G_b_matrix.shape[1] < n_b_svd:
        print(f"Warning: r_b={n_b_svd} but only {G_b_matrix.shape[1]} g_b component(s) were built "
              f"(check r_b classifications). Capping to {G_b_matrix.shape[1]}.")
    n_b_svd = min(n_b_svd, G_b_matrix.shape[1])

    if _g_store_list:
        _g_store_list[-1]['pred_G']      = np.array(predicted_g[0])   if predicted_g.size   > 0 else None
        _g_store_list[-1]['pred_G_b']    = np.array(predicted_g_b[0]) if predicted_g_b.size > 0 else None
        _sy43 = globals().get('start_year')
        _g_store_list[-1]['label'] = (
            str(_sy43 + m) if globals().get('style') == 'standard' and _sy43 is not None
            else f'm={m}')
        if center == 'column':
            _g_store_list[-1]['pred_X_mean'] = X_mean_pred
        elif center == 'row':
            _g_store_list[-1]['pred_X_mean'] = np.full(n, row_mean_pred)
        elif center == 'double':
            _g_store_list[-1]['pred_X_mean'] = X_mean_pred
        else:
            _g_store_list[-1]['pred_X_mean'] = None

    a_l    = (sum(predicted_g[0][j]   * Vt_rn[j, :] for j in range(r))
              if r       > 0 else np.zeros(n))
    B_pred = (sum(predicted_g_b[0][j] * Vt_b[j, :]  for j in range(n_b_svd))
              if n_b_svd > 0 else np.zeros(n))

    if n_b_svd > 0:
        c_b = c_list[-1] if c_list else 0.0
        if transform == 'log':
            A_N_pred = np.maximum(np.exp(B_pred) - c_b, 0.0)
        elif transform == 'sqrt':
            A_N_pred = np.maximum(B_pred ** 2 - c_b, 0.0)
        elif transform == 'cbrt':
            A_N_pred = B_pred ** 3 - c_b
        elif transform == 'power':
            A_N_pred = (np.sign(B_pred) * np.abs(B_pred) ** (1.0 / power_p) - c_b)
        else:
            A_N_pred = np.maximum(np.exp(B_pred) - c_b, 0.0)
    else:
        A_N_pred = np.exp(B_pred)

    _grand_mean_store = _g_store_list[-1].get('grand_mean', 0.0) if _g_store_list else 0.0
    prediction = np.atleast_1d(a_l) + np.atleast_1d(A_N_pred)
    if center == 'column':
        prediction = prediction + X_mean_pred
    elif center == 'row':
        prediction = prediction + row_mean_pred
    elif center == 'double':
        prediction = prediction + X_mean_pred + row_mean_pred - _grand_mean_store

    if _g_store_list:
        _g_store_list[-1]['decomp_a_l']    = np.atleast_1d(a_l).copy()
        _g_store_list[-1]['decomp_A_N']    = np.atleast_1d(A_N_pred).copy()
        if center == 'column':
            _g_store_list[-1]['decomp_X_mean'] = X_mean_pred.copy()
        elif center == 'row':
            _g_store_list[-1]['decomp_X_mean'] = np.full(n, row_mean_pred)
        elif center == 'double':
            _g_store_list[-1]['decomp_X_mean'] = X_mean_pred + row_mean_pred - _grand_mean_store
        else:
            _g_store_list[-1]['decomp_X_mean'] = np.zeros(n)
        if center in ('row', 'double'):
            _g_store_list[-1]['yearly_means']  = yearly_means
            _g_store_list[-1]['row_mean_pred'] = row_mean_pred
            _g_store_list[-1]['emp_row_mean']  = emp_row_mean
        else:
            _g_store_list[-1]['yearly_means']  = None
            _g_store_list[-1]['row_mean_pred'] = None
            _g_store_list[-1]['emp_row_mean']  = None
        _g_store_list[-1]['decomp_m']      = m
    return prediction.ravel()



def one_period_prediction_global_svd(
        m_hat, m, classifications, n, export_vector,
        g_orders=None, g_b_orders=None,
        g_forecaster=None, g_b_forecaster=None,
        transform='log', power_p=0.5,
        r_b=None, center='none',
        row_mean_order=(1, 1, 0),
        row_mean_forecaster=None,
        full_series=None):
    """
    Alternativa k one_period_prediction_classified.

    Vykona jednu globalnu SVD celej m-rocnej treningovej matice na ziskanie
    globalnych smerov Vt_global (namiesto SVD na kazdom okne). G-serie su
    projekcie centrovaného riadka kazdeho roka na tieto pevne smery:

        G[k - m_hat, i] = Xc[k, :] @ Vt_global[r_n_indices[i], :]

    Pre neregularnu (log-priestorovu) cast sa A_N zostavi z r_b_indices komponentov
    globalnej SVD; B = transformacia(A_N) ma vlastnu globalnu SVD (Vt_b_global)
    a projekcie riadkov B tvoria G_b. Klasifikacie su rešpektovane rovnako
    ako v one_period_prediction_classified.

    Centrovanie:
        'none'   - bez centrovania.
        'column' - odcita globalne mesacne priemery (axis=0); pri rekonstrukcii
                   sa konstantne pripoocitaju (bez potreby ARIMA).
        'row'    - odcita rocne priemery (axis=1); row_mean_forecaster predikuje
                   dalsi rocny priemer.
        'double' - DSVD (Zhang): odcita col_mean + row_mean - grand_mean;
                   rekonstrukcia = SVD + col_mean + row_mean_pred - grand_mean.

    row_mean_forecaster:
        None       -> ARIMA s pevnym radom row_mean_order (predvolene)
        'linear'   -> linearna extrapolacia trendu
        callable   -> series -> float
    """
    epsilon = 1e-10
    r_n_indices, r_b_indices, _ = _get_indices(classifications)
    r       = len(r_n_indices)
    n_b_svd = r_b if r_b is not None else len(r_b_indices)

    # -- 1. Zostavenie a centrovanie celej treningovej matice --
    vec    = np.asarray(export_vector, dtype=float)
    X_full = vec[:m * n].reshape(m, n)

    if center == 'column':
        X_mean    = X_full.mean(axis=0)          # (n,) global monthly means
        Xc        = X_full - X_mean
    elif center == 'row':
        row_means = X_full.mean(axis=1)          # (m,) yearly means
        Xc        = X_full - row_means[:, np.newaxis]
    elif center == 'double':
        X_mean    = X_full.mean(axis=0)
        row_means = X_full.mean(axis=1)
        grand_mean_val = float(X_full.mean())
        Xc        = X_full - X_mean - row_means[:, np.newaxis] + grand_mean_val
    else:
        X_mean = np.zeros(n)
        Xc     = X_full.copy()

    # -- 2. Globalna SVD treningovej matice --
    U_g, s_g, Vt_g = svd(Xc, full_matrices=False)
    # Stabilizacia znamienka
    _max_abs = np.argmax(np.abs(Vt_g), axis=1)
    _signs   = np.sign(Vt_g[np.arange(len(_max_abs)), _max_abs])
    Vt_g *= _signs[:, np.newaxis]
    U_g  *= _signs[np.newaxis, :]

    # -- 3. G_matrix: projekcia kazdeho rocneho riadka na globalne r_n smery --
    G_rows = []
    for k in range(m_hat, m):
        row    = Xc[k, :]
        g_vals = [float(row @ Vt_g[i, :]) for i in r_n_indices]
        G_rows.append(g_vals)
    G_matrix = np.array(G_rows) if G_rows else np.zeros((0, max(r, 1)))

    # -- 4. G_b_matrix: globalne projekcie B-priestoru --
    G_b_matrix = np.zeros((max(m - m_hat, 0), 0))
    Vt_b       = np.zeros((0, n))
    c_val      = 0.0

    if r_b_indices and n_b_svd > 0:
        A_N    = sum(s_g[i] * np.outer(U_g[:, i], Vt_g[i, :]) for i in r_b_indices)
        min_AN = float(np.min(A_N))
        c_val  = (-min_AN + epsilon) if min_AN <= 0 else epsilon
        f_tr   = _make_transform(transform, power_p, epsilon)
        B      = f_tr(A_N, c_val)               # (m, n)

        U_b, s_b, Vt_b_full = svd(B, full_matrices=False)
        Vt_b = Vt_b_full[:n_b_svd, :]

        G_b_rows = []
        for k in range(m_hat, m):
            b_row    = B[k, :]
            g_b_vals = [float(b_row @ Vt_b_full[i, :]) for i in range(n_b_svd)]
            G_b_rows.append(g_b_vals)
        G_b_matrix = np.array(G_b_rows) if G_b_rows else np.zeros((0, n_b_svd))

    # -- 5. Predikcia G a G_b --
    predicted_g, predicted_g_b = predict_g_vectors(
        G_matrix, G_b_matrix,
        g_orders=g_orders, g_b_orders=g_b_orders,
        g_forecaster=g_forecaster, g_b_forecaster=g_b_forecaster,
    )

    # -- 6. Predikcia centrovacieho clena --
    X_mean_add   = np.zeros(n)
    row_mean_add = 0.0
    grand_mean_sub = 0.0
    if center == 'column':
        X_mean_add = X_mean                     # constant - no ARIMA needed
    elif center in ('row', 'double'):
        try:
            fc_rm = _resolve_forecaster(row_mean_forecaster)
            if fc_rm is not None:
                row_mean_add = float(fc_rm(row_means))
            else:
                row_mean_add = float(
                    ARIMA(row_means, order=row_mean_order).fit().forecast(steps=1)[0])
        except Exception:
            row_mean_add = float(row_means[-1])
        if center == 'double':
            X_mean_add    = X_mean
            grand_mean_sub = grand_mean_val

    # -- 7. Rekonstrukcia --
    Vt_rn = Vt_g[r_n_indices, :] if r_n_indices else np.zeros((0, n))

    a_l    = (sum(float(predicted_g[0][j])   * Vt_rn[j, :] for j in range(r))
              if r       > 0 else np.zeros(n))
    B_pred = (sum(float(predicted_g_b[0][j]) * Vt_b[j, :]  for j in range(n_b_svd))
              if n_b_svd > 0 and Vt_b.shape[0] >= n_b_svd else np.zeros(n))

    if n_b_svd > 0 and r_b_indices:
        if transform == 'log':
            A_N_pred = np.maximum(np.exp(B_pred) - c_val, 0.0)
        elif transform == 'sqrt':
            A_N_pred = np.maximum(B_pred ** 2 - c_val, 0.0)
        elif transform == 'cbrt':
            A_N_pred = B_pred ** 3 - c_val
        elif transform == 'power':
            A_N_pred = np.sign(B_pred) * np.abs(B_pred) ** (1.0 / power_p) - c_val
        else:
            A_N_pred = np.maximum(np.exp(B_pred) - c_val, 0.0)
    else:
        A_N_pred = np.zeros(n)

    prediction = np.atleast_1d(a_l) + np.atleast_1d(A_N_pred) + X_mean_add + row_mean_add - grand_mean_sub

    # -- Empiricky rocny priemer nasledujuceho roka (pre vizualizaciu) --
    emp_row_mean = None
    if center in ('row', 'double'):
        _full = np.asarray(full_series, dtype=float) if full_series is not None else vec
        s_e, e_e = m * n, (m + 1) * n
        if e_e <= len(_full):
            emp_row_mean = float(_full[s_e:e_e].mean())

    # -- Empiricke g hodnoty pre GQRS-PM (projekcia nasledujuceho roka) --
    _full_gsv = np.asarray(full_series if full_series is not None else export_vector, dtype=float)
    _emp_G_gsv   = None
    _emp_G_b_gsv = None
    if len(_full_gsv) >= (m + 1) * n:
        _next_raw = _full_gsv[m * n : (m + 1) * n].astype(float)
        if center == 'column':
            _next_c = _next_raw - X_mean
        elif center == 'row':
            _next_c = _next_raw - float(_next_raw.mean())
        elif center == 'double':
            _next_c = _next_raw - X_mean - float(_next_raw.mean()) + grand_mean_val
        else:
            _next_c = _next_raw.copy()
        if r_n_indices:
            _emp_G_gsv = np.array([float(_next_c @ Vt_g[i, :]) for i in r_n_indices])
        else:
            _emp_G_gsv = np.zeros(0)
        if r_b_indices and n_b_svd > 0 and Vt_b.shape[0] >= n_b_svd:
            _a_n_next = sum(float(_next_c @ Vt_g[i, :]) * Vt_g[i, :] for i in r_b_indices)
            _b_next   = f_tr(np.asarray(_a_n_next, dtype=float), c_val)
            _emp_G_b_gsv = np.array([float(_b_next @ Vt_b[j, :]) for j in range(n_b_svd)])

    # -- Uloz rozklad pre plot_stored_decomposition --
    _g_store_list.append({
        'G': G_matrix, 'G_b': G_b_matrix,
        'pred_G':   np.array(predicted_g[0])   if predicted_g.size   > 0 else None,
        'pred_G_b': np.array(predicted_g_b[0]) if predicted_g_b.size > 0 else None,
        'emp_G': _emp_G_gsv, 'emp_G_b': _emp_G_b_gsv,
        'label': (str(globals().get('start_year', 0) + m)
                  if globals().get('style') == 'standard' and globals().get('start_year') is not None
                  else f'GlobalSVD m={m}'),
        'X_mean_matrix': None, 'emp_X_mean': None, 'pred_X_mean': None,
        'decomp_a_l':    np.atleast_1d(a_l).copy(),
        'decomp_A_N':    np.atleast_1d(A_N_pred).copy(),
        'decomp_X_mean': (np.atleast_1d(X_mean_add) + row_mean_add * np.ones(n) - grand_mean_sub * np.ones(n)).copy(),
        'decomp_m': m,
        'yearly_means':  row_means.tolist() if center in ('row', 'double') else None,
        'row_mean_pred': row_mean_add        if center in ('row', 'double') else None,
        'emp_row_mean':  emp_row_mean,
        'grand_mean':    grand_mean_val if center == 'double' else 0.0,
    })
    return prediction.ravel()



def sarima_12m_forecast(data_vector, seasonal_period=12):
    series = np.asarray(data_vector, dtype="float64")
    series = series[np.isfinite(series)]
    if len(series) < 2 * seasonal_period:
        raise ValueError(f"Need at least {2 * seasonal_period} observations")
    model = auto_arima(series, seasonal=True, m=seasonal_period,
                       stepwise=True, suppress_warnings=True,
                       error_action="ignore", trace=False)
    return np.asarray(model.predict(n_periods=seasonal_period), dtype="float64")



def mean_squared_error(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.mean((y_true - y_pred) ** 2)

def choose_tau_from_baseline(vec, quantile):
    vec = np.asarray(vec, float)
    return np.quantile(np.abs(vec[1:] - vec[:-1]), quantile)

def mean_capped_linear_error(y_true, y_pred, tol=0.05):
    """MCLE – relatívna abs. chyba ohraničená na 1. Tolerančné pásmo ±tol (default 5%).
    Skóre 0 = presná trefa, 1 = chyba  tol*|actual|, výsledok v [0, 1]."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.where(np.abs(y_true) > 0, tol * np.abs(y_true), np.nan)
    return float(np.nanmean(np.minimum(np.abs(y_true - y_pred) / denom, 1.0)))

def color_log(x):
    if x > 0:
        return f"\033[91m{x:+.2f}\033[0m"   # red
    elif x < 0:
        return f"\033[92m{x:+.2f}\033[0m"   # green
    else:
        return f"{x:+.2f}"
def mase(actual, predicted, training, s=12):
    """MASE – Hyndman & Koehler (2006), seasonal naive denominator from training only."""
    actual    = np.asarray(actual,    float)
    predicted = np.asarray(predicted, float)
    training  = np.asarray(training,  float)
    n = len(training)
    if n <= s:
        return float('nan')
    diffs = np.abs(training[s:] - training[:-s])
    D = np.mean(diffs)
    if D == 0:
        import warnings
        warnings.warn('MASE: denominator zero (flat series), excluding window')
        return float('nan')
    return float(np.mean(np.abs(actual - predicted)) / D)





from scipy import stats as _stats
import matplotlib.pyplot as _plt

def check_residual_normality(export_vector, prediction, n, m, alpha=0.05):
    """
    Testuje normalitu rezidui walk-forward predikcie pomocou Shapiro-Wilk testu.

    H0: rezidua pochádzajú z normálneho rozdelenia.
    Shapiro-Wilk má väčšiu silu testu pri malých vzorkách (n < 50) ako Jarque-Bera.

    Parameters
    ----------
    export_vector : list | np.ndarray
    prediction    : list of arrays  (výstup walk-forward slučky, index 0 = najnovšie)
    n             : int  (dĺžka periódy, zvyčajne 12)
    m             : int  (celkový počet úplných rokov v sérii)
    alpha         : float  (hladina významnosti, default 0.05)
    """
    p_windows = len(prediction)
    residuals = []
    for i in range(p_windows):
        actual = np.asarray(export_vector[n * (m - (i + 1)) : n * (m - i)], float)
        pred   = np.asarray(prediction[i], float)
        residuals.extend(actual - pred)
    residuals = np.array(residuals)

    sw_stat, sw_p  = _stats.shapiro(residuals)
    skewness       = _stats.skew(residuals)
    excess_kurt    = _stats.kurtosis(residuals)  # Fisher: 0 pre normálne

    # –– Vizualizácia ––––––––––––––––––––––––––––––––––––––––––––––––––––––––
    fig, (ax_qq, ax_hist) = _plt.subplots(1, 2, figsize=(12, 4))

    _stats.probplot(residuals, dist='norm', plot=ax_qq)
    ax_qq.set_title('Q-Q plot', fontsize=11)
    ax_qq.set_xlabel('Teoretické kvantily')
    ax_qq.set_ylabel('Vzorkové kvantily')
    ax_qq.get_lines()[0].set(markersize=4, alpha=0.6, color='#2c3e50')
    ax_qq.get_lines()[1].set(color='#e74c3c', linewidth=1.5)
    ax_qq.grid(True, alpha=0.3)

    ax_hist.hist(residuals, bins='auto', density=True,
                 color='#3498db', alpha=0.65, edgecolor='white', label='Rezidua')
    x_fit = np.linspace(residuals.min(), residuals.max(), 200)
    ax_hist.plot(x_fit,
                 _stats.norm.pdf(x_fit, residuals.mean(), residuals.std(ddof=1)),
                 color='#e74c3c', linewidth=2, label='N(μ, f²)')
    ax_hist.set_title('Histogram reziduí', fontsize=11)
    ax_hist.set_xlabel('Reziduál')
    ax_hist.set_ylabel('Hustota')
    ax_hist.legend(fontsize=9)
    ax_hist.grid(True, alpha=0.3)

    _plt.suptitle(
        f'Test normality reziduí  ({p_windows} okien, {len(residuals)} obs)',
        fontsize=12, fontweight='bold'
    )
    _plt.tight_layout()
    _plt.show()

    # –– Výpis výsledkov –––––––––––––––––––––––––––––––––––––––––––––––––––––
    print('Shapiro-Wilk test normality reziduí')
    print(f'  Pocet rezidui : {len(residuals)}')
    print(f'  Stredna hodnota  : {residuals.mean():.4f}')
    print(f'  Smerodajná odch. : {residuals.std(ddof=1):.4f}')
    print(f'  Sikmosť          : {skewness:.4f}')
    print(f'  Excesová spicatosť: {excess_kurt:.4f}  (0 = normálna)')
    print(f'  SW štatistika    : {sw_stat:.4f}')
    print(f'  p-hodnota        : {sw_p:.4f}')
    print()
    if sw_p > alpha:
        print(f'  -> H0 sa NEZAMIETA: rezidua sú zlucitelné s normalitou'
              f' (p = {sw_p:.4f} > alpha = {alpha})')
    else:
        print(f'  -> H0 sa ZAMIETA: rezidua vykazuju odchylku od normality'
              f' (p = {sw_p:.4f} <= alpha = {alpha})')
        if abs(skewness) > 0.5:
            print(f'     Prícina: výrazná šikmosť ({skewness:.3f})')
        if abs(excess_kurt) > 1.0:
            print(f'     Prícina: výrazná excesová špicatosť ({excess_kurt:.3f})')

    # –– Wilcoxonov test so znamienkami (H0: mu = 0, bez normality) –––––––––
    print()
    print('Wilcoxonov test so znamienkami  (H0: stredná hodnota = 0)')
    if len(residuals) < 10:
        print('  prilis malo pozorovaní, test sa preskocí (n < 10)')
    else:
        try:
            wx_stat, wx_p = _stats.wilcoxon(residuals, alternative='two-sided')
            print(f'  W štatistika : {wx_stat:.4f}')
            print(f'  p-hodnota    : {wx_p:.4f}')
            if wx_p > alpha:
                print(f'  -> H0 sa NEZAMIETA: stredná hodnota je zlucitelná s nulou'
                      f' (p = {wx_p:.4f} > alpha = {alpha})')
            else:
                print(f'  -> H0 sa ZAMIETA: stredná hodnota sa líší od nuly'
                      f' (p = {wx_p:.4f} <= alpha = {alpha})')
        except ValueError as _wx_err:
            print(f'  Wilcoxon: {_wx_err}')


def plot_mse_comparison(export_vector, chosen_prediction, sarima_prediction,
                        n=12, m=None, tau=None):
    """
    Skupinový stĺpcový graf MSE a relatívnych metrík (rMSE, rMCLE) na walk-forward okno.
    Ak je tau zadané, zobrazí aj rMCLE v druhom paneli.
    """
    if m is None:
        m = len(export_vector) // n

    p = min(len(chosen_prediction), len(sarima_prediction))

    mse_chosen_list, mse_sar_list = [], []
    mcle_chosen_list, mcle_sar_list = [], []
    all_test, all_chosen, all_sar = [], [], []
    window_labels = []

    for i in range(p):
        test = export_vector[n * (m - (i + 1)) : n * (m - i)]
        all_test.append(test)
        all_chosen.append(chosen_prediction[i])
        all_sar.append(sarima_prediction[i])
        mse_chosen_list.append(mean_squared_error(test, chosen_prediction[i]))
        mse_sar_list.append(mean_squared_error(test, sarima_prediction[i]))
        mcle_chosen_list.append(mean_capped_linear_error(test, chosen_prediction[i]))
        mcle_sar_list.append(mean_capped_linear_error(test, sarima_prediction[i]))
        window_labels.append(f'Window {i+1}\n(perióda {m - i})')

    cat_test   = np.concatenate(all_test)
    cat_chosen = np.concatenate(all_chosen)
    cat_sar    = np.concatenate(all_sar)
    mse_comb_chosen = mean_squared_error(cat_test, cat_chosen)
    mse_comb_sar    = mean_squared_error(cat_test, cat_sar)
    mcle_comb_chosen = mean_capped_linear_error(cat_test, cat_chosen)
    mcle_comb_sar    = mean_capped_linear_error(cat_test, cat_sar)

    mse_chosen_arr = np.array(mse_chosen_list)
    mse_sar_arr    = np.array(mse_sar_list)
    rmse_arr       = mse_chosen_arr / mse_sar_arr
    rmse_comb      = mse_comb_chosen / mse_comb_sar
    if tau is not None:
        rmcle_arr  = np.array(mcle_chosen_list) / np.array(mcle_sar_list)
        rmcle_comb = mcle_comb_chosen / mcle_comb_sar

    x         = np.arange(p)
    x_overall = p + 0.6
    width     = 0.35

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(8, (p + 1.8) * 2.5), 11))

    # –– Panel 1: absolútna MSE –––––––––––––––––––––––––––––––––––––––––––––
    bars_chosen = ax1.bar(
        x - width/2, mse_chosen_arr, width,
        label='Zvolený model', color='#e74c3c', alpha=0.85, edgecolor='white'
    )
    bars_sar = ax1.bar(
        x + width/2, mse_sar_arr, width,
        label='SARIMA', color='#3498db', alpha=0.85, edgecolor='white'
    )

    for bars, color in [(bars_chosen, '#e74c3c'), (bars_sar, '#3498db')]:
        for bar in bars:
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f'{bar.get_height():.0f}',
                ha='center', va='bottom', fontsize=8, color=color
            )

    ax1.bar(x_overall - width/2, mse_comb_chosen, width,
            color='#e74c3c', alpha=0.55, edgecolor='#e74c3c',
            linewidth=1.5, hatch='//', label='Zvolený model (celkovo)')
    ax1.bar(x_overall + width/2, mse_comb_sar, width,
            color='#3498db', alpha=0.55, edgecolor='#3498db',
            linewidth=1.5, hatch='//', label='SARIMA (celkovo)')

    for val, xc, color in [
        (mse_comb_chosen, x_overall - width/2, '#e74c3c'),
        (mse_comb_sar,    x_overall + width/2, '#3498db'),
    ]:
        ax1.text(xc, val * 1.01, f'{val:.0f}',
                 ha='center', va='bottom', fontsize=9, color=color, fontweight='bold')

    ax1.axvline(x=p - 0.15, color='gray', linestyle=':', linewidth=1.2, alpha=0.6)
    for val, color, ls in [
        (mse_chosen_arr.mean(), '#e74c3c', '--'),
        (mse_sar_arr.mean(),    '#3498db', ':'),
    ]:
        ax1.axhline(val, color=color, linewidth=1.0, linestyle=ls, alpha=0.5)

    ax1.set_xticks(list(x) + [x_overall])
    ax1.set_xticklabels(window_labels + [f'Overall\n({p} windows)'], fontsize=9)
    ax1.set_ylabel('MSE', fontsize=11)
    ax1.set_title(
        'MSE na okno doprednej validácie: Zvolený model vs SARIMA',
        fontsize=13, fontweight='bold'
    )
    ax1.legend(fontsize=9)
    ax1.grid(True, axis='y', alpha=0.3)

    # –– Panel 2: relatívne metriky (rMSE, rMCLE) ––––––––––––––––––––––––––
    has_mcle = True
    w2 = width * 0.75 if has_mcle else width
    off_mse  = -w2 / 2 if has_mcle else 0.0
    off_mcle =  w2 / 2

    ax2.bar(x + off_mse, rmse_arr, w2,
            label='rMSE', color='#e74c3c', alpha=0.85, edgecolor='white')
    if has_mcle:
        ax2.bar(x + off_mcle, rmcle_arr, w2,
                label='rMCLE', color='#9b59b6', alpha=0.85, edgecolor='white')

    ax2.bar(x_overall + off_mse, rmse_comb, w2,
            color='#e74c3c', alpha=0.55, edgecolor='#e74c3c',
            linewidth=1.5, hatch='//', label='rMSE (celkovo)')
    if has_mcle:
        ax2.bar(x_overall + off_mcle, rmcle_comb, w2,
                color='#9b59b6', alpha=0.55, edgecolor='#9b59b6',
                linewidth=1.5, hatch='//', label='rMCLE (celkovo)')

    for i2, val in enumerate(list(rmse_arr) + [rmse_comb]):
        xpos = (x[i2] if i2 < p else x_overall) + off_mse
        ax2.text(xpos, val * 1.01, f'{val:.3f}',
                 ha='center', va='bottom', fontsize=8, color='#e74c3c')
    if has_mcle:
        for i2, val in enumerate(list(rmcle_arr) + [rmcle_comb]):
            xpos = (x[i2] if i2 < p else x_overall) + off_mcle
            ax2.text(xpos, val * 1.01, f'{val:.3f}',
                     ha='center', va='bottom', fontsize=8, color='#9b59b6')

    ax2.axhline(1.0, color='black', linewidth=1.5, linestyle='--', alpha=0.7,
                label='SARIMA (referencia = 1)')
    ax2.axvline(x=p - 0.15, color='gray', linestyle=':', linewidth=1.2, alpha=0.6)
    ax2.set_xticks(list(x) + [x_overall])
    ax2.set_xticklabels(window_labels + [f'Overall\n({p} windows)'], fontsize=9)
    ax2.set_ylabel('Relatívna metrika (model / SARIMA)', fontsize=11)
    ax2.set_title(
        'rMSE' + (' a rMCLE' if has_mcle else '') + ' – relatívne voči SARIMA',
        fontsize=13, fontweight='bold'
    )
    ax2.legend(fontsize=9)
    ax2.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()



def load_multivariate_dataset(path):
    """
    Načíta CSV s viacerými krajinami (prvý stĺpec = mesiac/dátum, zvyšok = krajiny).
    Vráti dict {kód_krajiny: np.ndarray}.
    """
    df = pd.read_csv(path)
    # Vynechaj stĺpec dátumu/mesiaca
    if df.columns[0].lower() in ('month', 'date', 'time', 'year'):
        df = df.iloc[:, 1:]
    return {col: df[col].values.astype(float) for col in df.columns}


_sarima_cache = {}  # {series_hash: [predictions]}

def run_comparison(series, classifications,
                   n=12, m_hat=4, p=4,
                   g_orders=None, g_b_orders=None,
                   g_forecaster=None, g_b_forecaster=None,
                   transform='log', power_p=0.5,
                   r_b=1,
                   sarima_predictions=None,
                   run_sarima=True,
                   title='Porovnanie modelov',
                   center='none',
                   column_mean_order=(1, 1, 0),
                   row_mean_order=(1, 1, 0),
                   model='qspm',
                   row_mean_forecaster=None,
                   verbose=True,
                   tol=0.05):
    """
    Walk-forward porovnanie QS-PM vs SARIMA na zadanej sérii.

    Parametre
    ---------
    series             : 1-D array-like
    classifications    : dict z classify_eigenvalues_interactive
    n                  : dĺžka periódy (default 12)
    m_hat              : veľkosť rolovacieho okna v periódach (default 4)
    p                  : počet walk-forward testovacích okien (default 4)
    sarima_predictions : predpočítaný zoznam SARIMA - preskočí auto_arima ak je zadaný
    run_sarima         : False = úplne preskočí SARIMA (vráti None pre sar)
    title              : reťazec zobrazený v názvoch grafov a konzole

    Vráti
    -----
    dict: svd_arima, sarima, mse_svd, mse_sarima, series, m
    """
    import matplotlib.pyplot as plt
    series = np.asarray(series, dtype=float)
    m = len(series) // n

    # -- Walk-forward --––––
    import sys as _sys
    _model_label = 'GQRS-PM' if model == 'gqrspm' else 'QS-PM'
    clear_g_store()
    svd_preds = []
    for i in range(p):
        cut = series[:n * (m - (i + 1))]
        _sys.stdout.write(f'\r  {_model_label} okno {i+1}/{p}...')
        _sys.stdout.flush()
        if model == 'gqrspm':
            pred = one_period_prediction_global_svd(
                m_hat, m - (i + 1), classifications, n, cut,
                g_orders=g_orders, g_b_orders=g_b_orders,
                g_forecaster=g_forecaster, g_b_forecaster=g_b_forecaster,
                transform=transform, power_p=power_p,
                r_b=r_b, center=center,
                row_mean_order=row_mean_order,
                row_mean_forecaster=row_mean_forecaster, full_series=cut)
        else:
            pred = one_period_prediction_classified(
                m_hat, m - (i + 1), classifications, n, cut,
                g_orders=g_orders, g_b_orders=g_b_orders,
                g_forecaster=g_forecaster, g_b_forecaster=g_b_forecaster,
                transform=transform, power_p=power_p,
                r_b=r_b,
                center=center, column_mean_order=column_mean_order,
                row_mean_order=row_mean_order, full_series=series)
        svd_preds.append(pred)
    _sys.stdout.write('\r' + ' ' * 40 + '\r')
    _sys.stdout.flush()


    # -- SARIMA walk-forward --––––
    _cache_key = hash(series.tobytes())
    if sarima_predictions is not None:
        sar_preds = list(sarima_predictions)
        _sarima_cache[_cache_key] = sar_preds
    elif _cache_key in _sarima_cache:
        if verbose:
            print("Pouzivam ulozene SARIMA predikcie z cache.")
        sar_preds = _sarima_cache[_cache_key]
    elif run_sarima:
        if verbose:
            print("Running SARIMA (moze trvat dlhsie)...")
        sar_preds = []
        for i in range(p):
            cut = series[:n * (m - (i + 1))]
            sar_preds.append(sarima_12m_forecast(cut, seasonal_period=n))
            _sys.stdout.write(f'\r  SARIMA okno {i+1}/{p}...')
            _sys.stdout.flush()
        _sys.stdout.write('\r' + ' ' * 40 + '\r')
        _sys.stdout.flush()
        _sarima_cache[_cache_key] = sar_preds
        if verbose:
            print("SARIMA predikcie ulozene do cache.")
    else:
        sar_preds = None

    # -- Zbieranie spektra (singulárne hodnoty na walk-forward okno) --
    spectra = []
    for i in range(p):
        cut_w = series[:n * (m - (i + 1))]
        m_cut = len(cut_w) // n
        window_data = create_data_window_vector(cut_w, m_cut, n, m_hat)
        svd_res = process_vector_to_svd(window_data, center=center)
        spectra.append(svd_res['s'].copy())

    # -- Metriky --––––
    all_test, all_svd, all_naive = [], [], []
    mse_per_window = []
    mse_sar_per_window = []
    mase_per_window = []
    mase_sar_per_window = []
    rmcle_per_window = []
    rmcle_sar_per_window = []
    for i in range(p):
        actual     = series[n * (m - (i + 1))     : n * (m - i)]
        naive_i    = series[n * (m - (i + 1) - 1) : n * (m - (i + 1))]
        training_i = series[:n * (m - (i + 1))]
        all_test.append(actual)
        all_svd.append(svd_preds[i])
        all_naive.append(naive_i)
        mse_per_window.append(float(mean_squared_error(actual, svd_preds[i])))
        mase_per_window.append(mase(actual, svd_preds[i], training_i, s=n))
        _mcle_i  = float(mean_capped_linear_error(actual, svd_preds[i], tol=tol))
        _mcle_ni = float(mean_capped_linear_error(actual, naive_i, tol=tol))
        rmcle_per_window.append(_mcle_i / _mcle_ni if _mcle_ni > 0 else float('nan'))
        if sar_preds is not None:
            mse_sar_per_window.append(float(mean_squared_error(actual, sar_preds[i])))
            mase_sar_per_window.append(mase(actual, sar_preds[i], training_i, s=n))
            _mcle_si = float(mean_capped_linear_error(actual, sar_preds[i], tol=tol))
            rmcle_sar_per_window.append(_mcle_si / _mcle_ni if _mcle_ni > 0 else float('nan'))
    cat_test  = np.concatenate(all_test)
    cat_svd   = np.concatenate(all_svd)
    cat_naive = np.concatenate(all_naive)
    mse_svd   = mean_squared_error(cat_test, cat_svd)
    mase_svd  = float(np.nanmean(mase_per_window)) if mase_per_window else float('nan')
    rmcle_svd = float(np.nanmean(rmcle_per_window))

    mse_sar = None
    mase_sarima = None
    rmcle_sar = None
    if sar_preds is not None:
        cat_sar     = np.concatenate(sar_preds)
        mse_sar     = mean_squared_error(cat_test, cat_sar)
        mase_sarima = float(np.nanmean(mase_sar_per_window)) if mase_sar_per_window else float('nan')
        rmcle_sar   = float(np.nanmean(rmcle_sar_per_window))
    else:
        mse_sar_per_window = None

    mcle_svd = float(mean_capped_linear_error(cat_test, cat_svd, tol=tol))
    mcle_sar = float(mean_capped_linear_error(cat_test, cat_sar, tol=tol)) if sar_preds is not None else None
    if sar_preds is None:
        mase_sarima = None

    # -- Výpis zhrnutia --––––
    if verbose:
        print(f'\n=== {title} ===')
        print(f'Dlzka serie: {len(series)} obs  |  {m} period  |  {p} okien')
        print(f'{_model_label} combined MSE : {mse_svd:.6f}')
        if mse_sar is not None:
            print(f'SARIMA    combined MSE : {mse_sar:.6f}')
            print(f'Log-ratio (SVD/SARIMA): {np.log(mse_svd / mse_sar):+.4f}')

    # -- Graf predikcií (2 farby, všetky okná naraz) --––––
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(series, color='#2c3e50', linewidth=1.4, label='Skutočná séria')
    for i in range(p):
        ts = n * (m - (i + 1))
        te = n * (m - i)
        ax.axvspan(ts, te, alpha=0.10, color='gray', zorder=1)
        ax.plot(np.arange(ts, te), svd_preds[i],
                color='#e74c3c', linewidth=1.8, linestyle='--',
                label=_model_label if i == 0 else '_nolegend_')
        if sar_preds is not None:
            ax.plot(np.arange(ts, te), sar_preds[i],
                    color='#3498db', linewidth=1.8, linestyle='-.',
                    label='SARIMA' if i == 0 else '_nolegend_')
    ax.set_title(f'{title}  [{_model_label}]', fontsize=12, fontweight='bold')
    ax.set_xlabel('Index mesiaca')
    ax.set_ylabel('Hodnota')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return dict(svd_arima=svd_preds, sarima=sar_preds,
                mse_svd=mse_svd, mse_sarima=mse_sar,
                mase_svd=mase_svd, mase_sarima=mase_sarima,
                mcle_svd=mcle_svd, mcle_sarima=mcle_sar,
                rmcle_svd=rmcle_svd, rmcle_sarima=rmcle_sar,
                mse_per_window=mse_per_window,
                mse_sar_per_window=mse_sar_per_window,
                spectra=spectra,
                series=series, m=m)


def compare_datasets(dataset_dict, classifications,
                     n=12, m_hat=4, p=4,
                     g_orders=None, g_b_orders=None,
                     g_forecaster=None, g_b_forecaster=None,
                     transform='log', power_p=0.5,
                     r_b=1,
                     sarima_dict=None,
                     run_sarima=True,
                     sarima_pkl=None,
                     center='none',
                     column_mean_order=(1, 1, 0),
                     row_mean_order=(1, 1, 0),
                     model='qspm',
                     row_mean_forecaster=None,
                     tag=None,
                     tol=0.05):
    """
    Spustí run_comparison na viacerých datasetoch a vytvorí porovnávací
    graf MSE naprieč datasetmi.

    Parametre
    ---------
    dataset_dict  : dict {label: series}  - jeden záznam na dataset
    sarima_dict   : dict {label: sarima_predictions}  - predpočítaná SARIMA
                    na dataset (voliteľné, fallback na run_sarima)
    sarima_pkl    : cesta str - ak je zadaná, načíta SARIMA predikcie z pickle
                    pri prvom spustení a uloží ich po výpočte
    (ostatné argumenty sa odovzdajú do run_comparison)

    Vráti
    -----
    dict {label: result_dict from run_comparison}
    """
    import matplotlib.pyplot as plt
    import pickle, os

    # Načítaj predpočítané SARIMA z pkl, ak je dostupné
    if sarima_pkl and os.path.exists(sarima_pkl):
        with open(sarima_pkl, 'rb') as _f:
            sarima_dict = pickle.load(_f)
        print(f'Loaded SARIMA predictions from {sarima_pkl}')
    sarima_dict = sarima_dict or {}
    results = {}

    import sys as _sys
    _total_ds = len(dataset_dict)
    for _ds_idx, (label, series) in enumerate(dataset_dict.items()):
        _sys.stdout.write(f'\r  [{_ds_idx+1}/{_total_ds}] {label}...')
        _sys.stdout.flush()
        res = run_comparison(
            series, classifications,
            n=n, m_hat=m_hat, p=p,
            g_orders=g_orders, g_b_orders=g_b_orders,
            g_forecaster=g_forecaster, g_b_forecaster=g_b_forecaster,
            transform=transform, power_p=power_p,
            r_b=r_b,
            sarima_predictions=sarima_dict.get(label),
            run_sarima=run_sarima,
            title=label,
            center=center,
            column_mean_order=column_mean_order,
            row_mean_order=row_mean_order,
            model=model,
            row_mean_forecaster=row_mean_forecaster,
            verbose=False,
            tol=tol,
        )
        results[label] = res
    _sys.stdout.write('\r' + ' ' * 60 + '\r')
    _sys.stdout.flush()

    # -- Uloženie SARIMA predikcií do pkl --
    if sarima_pkl:
        _sar_to_save = {lbl: results[lbl]['sarima']
                        for lbl in results if results[lbl]['sarima'] is not None}
        if _sar_to_save:
            with open(sarima_pkl, 'wb') as _f:
                pickle.dump(_sar_to_save, _f)
            print(f'Saved SARIMA predictions to {sarima_pkl}')

    # -- Per-krajina tabulka a breakdown -----------------------------------
    labels = list(results.keys())
    _ml = 'GQRS-PM' if model == 'gqrspm' else 'QS-PM'
    _rows_br = []
    for l in labels:
        ms  = results[l]['mse_svd']
        msr = results[l]['mse_sarima']
        lr  = round(float(np.log(ms / msr)), 3) if msr else None
        _rmcle_sv = results[l]['rmcle_svd']
        _rmcle_sr = results[l]['rmcle_sarima']
        _mse_sv   = results[l]['mse_svd']
        _mse_sr   = results[l]['mse_sarima']
        _mase_sv  = results[l]['mase_svd']
        _mase_sr  = results[l]['mase_sarima']
        _mcle_sv  = results[l]['mcle_svd']
        _mcle_sr  = results[l]['mcle_sarima']
        def _lr(a, b): return round(float(np.log(a / b)), 4) if a and b else None
        _rows_br.append({'Krajina': l,
                         'MSE':  _lr(_mse_sv,  _mse_sr),
                         'MASE': _lr(_mase_sv, _mase_sr),
                         'MCLE': _lr(_mcle_sv, _mcle_sr)})
    _df_br = pd.DataFrame(_rows_br).set_index('Krajina')
    def _clr_br(v):
        try:
            fv = float(v)
            return 'color: #2a7a2a' if fv < 0 else 'color: #cc3333'
        except Exception:
            return ''
    # -- Kumulátívny riadok -----------------------------------------------
    _sl = list(_df_br.index)
    _has_sar = any(results[l]['sarima'] is not None for l in _sl)
    _c_lr = _c_lr_rmse = _c_lr_mcle = _c_lr_rmcle = None
    if _has_sar:
        _sar_sl = [l for l in _sl if results[l]['sarima'] is not None]
        def _lmean(key_a, key_b):
            a = np.mean([results[l][key_a] for l in _sar_sl if results[l][key_a] is not None])
            b = np.mean([results[l][key_b] for l in _sar_sl if results[l][key_b] is not None])
            return round(float(np.log(a / b)), 4) if b else None
        _c_lr       = _lmean('mse_svd',   'mse_sarima')
        _c_lr_mase  = _lmean('mase_svd',  'mase_sarima')
        _c_lr_mcle  = _lmean('mcle_svd',  'mcle_sarima')
        _c_lr_rmcle = _lmean('rmcle_svd', 'rmcle_sarima')
    _cum_row = pd.DataFrame(
        [{'MSE': _c_lr, 'MASE': _c_lr_mase,
          'MCLE': _c_lr_mcle}],
        index=['–– Celkovo ––'])
    _df_br = pd.concat([_df_br, _cum_row])

    _all_cols = [c for c in ['MSE', 'MASE', 'MCLE'] if c in _df_br.columns]
    from IPython.display import display as _disp_br
    _disp_br(_df_br.style
             .applymap(_clr_br, subset=_all_cols)
             .format({c: '{:+.4f}' for c in _all_cols}, na_rep='---')
             .set_caption(f'Per-krajina log(metrika_model/metrika_SARIMA) [{_ml}] | poradie: podla vypoctu'))

    # -- Automaticke ulozenie tabulky podla tagu -------------------------
    if tag:
        import os as _os
        _os.makedirs('outputs', exist_ok=True)
        _sp = f'outputs/exec_{tag}_results'
        _df_br.to_csv(f'{_sp}.csv')
        with open(f'{_sp}.pkl', 'wb') as _spf:
            import pickle as _spkl
            _spkl.dump(_df_br, _spf)
        print(f'Tabulka ulozena: {_sp}.csv / .pkl')

    # -- Agregat -------------------------------------------------------
    _all_mse_sv   = [results[l]['mse_svd']    for l in labels]
    _all_mse_sar  = [r for r in [results[l]['mse_sarima']  for l in labels] if r is not None]
    _all_mase_sv  = [results[l]['mase_svd']    for l in labels]
    _all_mase_sar = [r for r in [results[l]['mase_sarima'] for l in labels] if r is not None]
    _all_log_mase_sv  = [np.log(v) for v in _all_mase_sv  if v and not np.isnan(v)]
    _all_log_mase_sar = [np.log(v) for v in _all_mase_sar if v and not np.isnan(v)]
    print(f'\n=== Agregat ({len(labels)} krajin) ===')
    print(f'  Priemer MASE {_ml}  : {np.nanmean(_all_mase_sv):.4f}  (< 1 = lepsi ako sezonna naivna)')
    if _all_mase_sar:
        print(f'  Priemer MASE SARIMA : {np.nanmean(_all_mase_sar):.4f}')
    if _all_log_mase_sv and _all_log_mase_sar:
        print(f'  Priemer log(MASE/SAR) {_ml}: {np.mean([np.log(a/b) for a,b in zip(_all_mase_sv,_all_mase_sar) if a and b and not np.isnan(a) and not np.isnan(b)]):+.4f}')
    print(f'  Priemer MSE {_ml}  : {np.mean(_all_mse_sv):.4f}')
    print(f'  Median  MSE {_ml}  : {np.median(_all_mse_sv):.4f}')
    if _all_mse_sar:
        print(f'  Priemer MSE SARIMA : {np.mean(_all_mse_sar):.4f}')
        print(f'  Median  MSE SARIMA : {np.median(_all_mse_sar):.4f}')
        _lr_all = [np.log(results[l]['mse_svd'] / results[l]['mse_sarima'])
                   for l in labels if results[l]['mse_sarima']]
        _n_better = sum(1 for v in _lr_all if v < 0)
        print(f'  {_ml} lepsie v {_n_better}/{len(_lr_all)} krajinach')
        print(f'  Priemer log-pomer  : {np.mean(_lr_all):+.4f}')

    return results


def load_or_compute_sarima(series_dict, p, n=12, pkl_dir='data/lists', tag=None):
    """
    Nacita SARIMA predikcie pre kazdu sériu z pkl (ak existuju a maju >= p okien),
    inak ich vypocita a ulozi. Vracia dict {label: [p predikcii]}.

    Parametre
    ---------
    series_dict : dict {label: np.ndarray}
    p           : pocet walk-forward okien
    n           : dlzka periódy (default 12)
    pkl_dir     : adresar kde sa uklada/nacitava pkl
    tag         : identifikator datasetu (napr. '10c'); ak None, hash klucov

    Subor: {pkl_dir}/sarima_{tag}_p{p}.pkl
    Ak existuje pkl pre rovnaky tag a p >= pozadovane, iba nacita.
    Ak chybaju niektore serie alebo je p primas maly, dopocita a ulozi.
    """
    import pickle, os
    if tag is None:
        tag = str(abs(hash(tuple(sorted(series_dict.keys())))))[:8]
    pkl_path = os.path.join(pkl_dir, f'sarima_{tag}_p{p}.pkl')
    os.makedirs(pkl_dir, exist_ok=True)

    cache = {}
    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as _f:
            cache = pickle.load(_f)
        labels_ok = [lbl for lbl in series_dict
                     if lbl in cache and len(cache[lbl]) >= p]
        if len(labels_ok) == len(series_dict):
            print(f'SARIMA: nacitane z {pkl_path}  ({len(series_dict)} serii, p={p})')
            return {lbl: cache[lbl][:p] for lbl in series_dict}
        missing = [lbl for lbl in series_dict if lbl not in labels_ok]
        print(f'SARIMA: {pkl_path} neuplny (chybaju: {missing}), dopocitavam...')
    else:
        print(f'SARIMA: {pkl_path} neexistuje, pocitam...')

    for lbl, ser in series_dict.items():
        if lbl in cache and len(cache[lbl]) >= p:
            continue
        ser = np.asarray(ser, float)
        m_s = len(ser) // n
        preds = []
        for wi in range(p):
            cut = ser[:n * (m_s - (wi + 1))]
            preds.append(sarima_12m_forecast(cut, seasonal_period=n))
            print(f'  SARIMA {lbl}  okno {wi+1}/{p}', end='\r')
        cache[lbl] = preds
        with open(pkl_path, 'wb') as _f:
            pickle.dump(cache, _f)

    print(f'SARIMA ulozene: {pkl_path}')
    return {lbl: cache[lbl][:p] for lbl in series_dict}

In [69]:
def plot_singular_value_spectrum_3d(
        export_vector, m_hat, m, n=12,
        max_components=None, testset_size=None,
        classifications=None,
        start_year=None, center='none'):
    """
    3D čiarový graf spektra singulárnych hodnôt cez rolovacie okná.

    Každé okno = farebná čiara (sigma_0 >= sigma_1 >= ...).
    Staršie okná = tmavšie (plasma), novšie = svetlejšie.
    Popis osi Z je vykreslený cez fig.text() (obchádza orezy matplotlib 3D).

    Parametre
    ---------
    export_vector  : 1-D array-like
    m_hat          : int   - veľkosť rolovacieho okna (v periódach)
    m              : int   - celkový počet úplných periód
    n              : int   - dĺžka periódy (default 12)
    max_components : int alebo None - orezanie spektra
    testset_size   : int alebo None - počet posledných okien na zobrazenie
    classifications: dict alebo None - farby bodov: r_n=modrá, r_b=oranžová, redundant=šedá
    """
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.patches as mpatches

    # -- Zbieranie spektier --
    raw_spectra = []
    k_values = []
    for k in range(m_hat, m):
        window = create_data_window_vector(export_vector, k, n, m_hat)
        s = process_vector_to_svd(window, center=center)['s']
        if max_components is not None:
            s = s[:max_components]
        raw_spectra.append(s)
        k_values.append(k)
    if testset_size is not None:
        raw_spectra = raw_spectra[-testset_size:]
        k_values = k_values[-testset_size:]

    spectra = raw_spectra

    n_win   = len(spectra)
    n_comps = len(spectra[0])
    xs      = np.arange(n_comps)
    z_max   = max(s.max() for s in spectra)

    # -- Farby bodov komponentov podľa klasifikácií --
    comp_colors = None
    if classifications is not None:
        _role = {k: v for k, v in classifications.items() if isinstance(k, int)}
        pal = {'r_n': '#2979ff', 'r_b': '#ff6d00', 'redundant': '#9e9e9e'}
        comp_colors = [pal.get(_role.get(i, 'redundant'), '#9e9e9e')
                       for i in range(n_comps)]

    win_colors = cm.plasma(np.linspace(0.15, 0.85, n_win))

    # -- Graf --
    fig = plt.figure(figsize=(13, 8))
    ax  = fig.add_subplot(111, projection='3d')

    for wi, (s_vals, wc) in enumerate(zip(spectra, win_colors)):
        ax.plot(xs, [wi] * n_comps, s_vals,
                color=wc, linewidth=1.8, alpha=0.90)
        s0 = raw_spectra[wi][0]
        for xi, sv in enumerate(s_vals):
            dc = comp_colors[xi] if comp_colors else 'black'
            ax.scatter(xi, wi, sv, color='black', s=22, zorder=5, depthshade=False)
            s_sum = raw_spectra[wi].sum()
            ratio = raw_spectra[wi][xi] / s_sum if s_sum != 0 else float('nan')
            ax.text(xi, wi, sv + z_max * 0.04, f'{ratio:.2f}',
                    fontsize=12, fontweight='bold', ha='center', va='bottom', color='red')

    # -- Osi (bez zlabel - vykreslený cez fig.text) --
    ax.set_xlim(-0.8, n_comps - 0.2)
    ax.set_ylim(-0.8, n_win  - 0.2)
    ax.set_zlim(0, z_max * 1.25)

    ax.set_xticks(range(n_comps))
    ax.set_xticklabels([f'$\\sigma_{{{i}}}$' for i in range(n_comps)], fontsize=11)
    ax.set_yticks(range(n_win))
    if start_year is not None:
        ax.set_yticklabels([str(start_year + k) for k in k_values], fontsize=11)
    else:
        ax.set_yticklabels([str(i) for i in range(n_win)], fontsize=11)
    ax.set_xlabel('Komponent', labelpad=8, fontsize=13)
    ax.set_ylabel('Okno',      labelpad=8, fontsize=13)
    ax.set_zlabel('')   # suppress - replaced by fig.text below

    fig.text(0.12, 0.55, 'Singulárna hodnota $\sigma$', va='center', ha='center',
             rotation='vertical', fontsize=13)

    # -- Legenda --
    ax.legend(handles=[mpatches.Patch(color='red', label=r'$\sigma_i/\sum\sigma_i$ (pomer)')],
              fontsize=13, loc='upper left')

    ax.view_init(elev=30, azim=-40)
    plt.subplots_adjust(left=-0.25, right=0.95)
    plt.show()


def plot_nonregular_spectrum_3d(
        export_vector, m_hat, m, n=12,
        r=1, max_components=None, testset_size=None,
        transform='log', power_p=0.5, epsilon=1e-10,
        classifications=None,
        start_year=None, 
        center='none'):
    """
    3D graf spektra neregulárnej časti SVD rozkladu.

    Pre každé rolovacie okno:
      1. Zostav A_N = súčet rank-1 matíc pre komponenty r..p-1
      2. Aplikuj nelineárnu transformáciu B = f(A_N)
      3. SVD matice B -> singulárne hodnoty zobrazené v grafe

    Parametre
    ---------
    export_vector  : 1-D array-like
    m_hat          : int   - veľkosť rolovacieho okna (v periódach)
    m              : int   - celkový počet úplných periód
    n              : int   - dĺžka periódy (default 12)
    r              : int   - počet preskočených regulárnych komponentov (default 1)
    max_components : int alebo None - orezanie B-spektra
    testset_size   : int alebo None - počet posledných okien na zobrazenie
    transform      : str  - 'log' | 'sqrt' | 'cbrt' | 'power'
    epsilon        : float - offset pre numerickú stabilitu
    classifications: dict alebo None - farby bodov z classify_eigenvalues_interactive
    """
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.patches as mpatches

    TRANSFORMS = {
        'log':   lambda A, c: np.log(np.maximum(A + c, epsilon)),
        'sqrt':  lambda A, c: np.sqrt(np.maximum(A + c, 0.0)),
        'cbrt':  lambda A, c: np.cbrt(A + c),
        'power': lambda A, c: np.sign(A + c) * np.abs(A + c) ** power_p,
    }
    if transform not in TRANSFORMS:
        raise ValueError(f'Neznama transformacia: {transform!r}. '
                         f'Dostupne: {list(TRANSFORMS)}')
    f_transform = TRANSFORMS[transform]

    # -- Zbieranie B-spektier pre každé okno --
    raw_spectra = []
    k_values = []
    for k in range(m_hat, m):
        window = create_data_window_vector(export_vector, k, n, m_hat)
        res = process_vector_to_svd(window, center=center)
        U, s, Vt = res['U'], res['s'], res['Vt']
        p = min(U.shape[0], Vt.shape[1])

        if classifications is not None:
            r_b_indices = sorted(k for k, v in classifications.items() if isinstance(k, int) and v == "r_b")
            an_indices = [i for i in r_b_indices if i < p]
        else:
            an_indices = list(range(r, p))

        if not an_indices:
            raw_spectra.append(np.array([0.0]))
            k_values.append(k)
            continue

        A_N = sum(s[i] * U[:, i:i+1] @ Vt[i:i+1, :] for i in an_indices)
        min_AN = np.min(A_N)
        c = (-min_AN + epsilon) if min_AN <= 0 else epsilon
        B = f_transform(A_N, c)

        s_B = process_vector_to_svd(B.flatten())['s']
        if max_components is not None:
            s_B = s_B[:max_components]
        raw_spectra.append(s_B)
        k_values.append(k)

    if testset_size is not None:
        raw_spectra = raw_spectra[-testset_size:]
        k_values = k_values[-testset_size:]

    # Zarovnanie na rovnakú dĺžku
    n_comps = max(len(s) for s in raw_spectra)
    raw_spectra = [np.pad(s, (0, n_comps - len(s))) for s in raw_spectra]

    spectra = raw_spectra

    n_win = len(spectra)
    xs    = np.arange(n_comps)
    z_max = max(s.max() for s in spectra)

    # -- Farby bodov komponentov (z klasifikácií, ak sú zadané) --
    comp_colors = None
    if classifications is not None:
        _role = {k: v for k, v in classifications.items() if isinstance(k, int)}
        pal = {'r_n': '#2979ff', 'r_b': '#ff6d00', 'redundant': '#9e9e9e'}
        n_cls = min(n_comps, max((k for k in _role), default=-1) + 1)
        comp_colors = [pal.get(_role.get(i, 'redundant'), '#9e9e9e')
                       for i in range(n_comps)]

    win_colors = cm.plasma(np.linspace(0.15, 0.85, n_win))

    # -- Graf --
    fig = plt.figure(figsize=(13, 8))
    ax  = fig.add_subplot(111, projection='3d')

    for wi, (s_vals, wc) in enumerate(zip(spectra, win_colors)):
        ax.plot(xs, [wi] * n_comps, s_vals,
                color=wc, linewidth=1.8, alpha=0.90)
        s0 = raw_spectra[wi][0]
        for xi, sv in enumerate(s_vals):
            ax.scatter(xi, wi, sv, color='black', s=22, zorder=5, depthshade=False)
            s_sum = raw_spectra[wi].sum()
            ratio = raw_spectra[wi][xi] / s_sum if s_sum != 0 else float('nan')
            ax.text(xi, wi, sv + z_max * 0.04, f'{ratio:.2f}',
                    fontsize=13, fontweight='bold', ha='center', va='bottom', color='red')

    # -- Osi --
    ax.set_xlim(-0.8, n_comps - 0.2)
    ax.set_ylim(-0.8, n_win  - 0.2)
    ax.set_zlim(0, z_max * 1.25)

    ax.set_xticks(range(n_comps))
    ax.set_xticklabels([f'$\\sigma^B_{{{i}}}$' for i in range(n_comps)], fontsize=11)
    ax.set_yticks(range(n_win))
    if start_year is not None:
        ax.set_yticklabels([str(start_year + k) for k in k_values], fontsize=11)
    else:
        ax.set_yticklabels([str(i) for i in range(n_win)], fontsize=11)
    ax.set_xlabel('Komponent', labelpad=8, fontsize=13)
    ax.set_ylabel('Okno',      labelpad=8, fontsize=13)
    ax.set_zlabel('')

    zlabel = f'Sing. hodnota $\\sigma^B$ ({transform})'
    fig.text(0.12, 0.55, zlabel, va='center', ha='center',
             rotation='vertical', fontsize=13)

    # -- Legenda --
    handles = [mpatches.Patch(color='red', label=r'$\sigma^B_i/\sum\sigma^B_i$ (pomer)')]
    ax.legend(handles=handles, fontsize=13, loc='upper left')

    ax.view_init(elev=30, azim=-40)
    plt.subplots_adjust(left=-0.18, right=0.82)
    plt.show()



def plot_vt_stability(series_list, m_hat=4, n=12, r=2, center="row"):
    """
    Zmeria ||V_last - V_k||_F / sqrt(r) pre kazde rolovacie okno.

    series_list : list of (time_series, label)
    X-os: pocet okien od posledneho (0 = posledne okno, rastie dozadu).
    Normalizacia: Vt riadky su jednotkove => max diff = 2*sqrt(r) => / sqrt(r) => [0, 2].
    """
    COLORS = ["#c0392b", "#2471a3"]
    FS_TITLE  = 14
    FS_LABEL  = 13
    FS_TICK   = 12
    FS_LEGEND = 12
    LW, MS    = 2.2, 7

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 4))

    for (time_series, label), color in zip(series_list, COLORS):
        vec = np.asarray(time_series, dtype=float)
        m   = len(vec) // n

        svds = {}
        for k in range(m_hat, m + 1):
            window  = vec[(k - m_hat) * n : k * n]
            svds[k] = process_vector_to_svd(window, center=center)["Vt"][:r, :]

        Vt_last = svds[m]
        diffs   = np.array([
            np.linalg.norm(Vt_last - svds[k], "fro") / np.sqrt(r)
            for k in range(m_hat, m + 1)
        ])
        x = np.arange(len(diffs) - 1, -1, -1)

        ax.plot(x, diffs, marker="o", linewidth=LW, markersize=MS,
                color=color, label=label)

    ax.invert_xaxis()
    ax.set_xlabel("Počet okien od posledného", fontsize=FS_LABEL, fontweight="bold")
    ax.set_ylabel(
        r"$\|V_{\mathrm{last}} - V_k\|_F \;/\; \sqrt{r}$",
        fontsize=FS_LABEL, fontweight="bold")
    ax.set_title("Stabilita matice V naprieč oknami",
                 fontsize=FS_TITLE, fontweight="bold")
    ax.set_ylim(bottom=0)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.tick_params(axis="both", labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND, framealpha=0.7)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()



import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


def plot_autocorrelation(time_series, n=12, lags=48, show_pacf=True):
    """
    Vykreslí ACF a PACF mesačného časového radu, každú v samostatnom grafe.
    Výpočet použije celú sériu; lags riadi len počet zobrazených lagov.

    Parametre
    ---------
    time_series : array-like - vstupný časový rad
    n           : int        - sezónna perióda (default 12); zvislé čiary pri násobkoch n
    lags        : int        - počet lagov na zobrazenie (default 48)
    """
    series    = np.asarray(time_series, dtype="float64")
    lags      = min(lags, len(series) - 1)
    pacf_lags = min(lags, len(series) // 2 - 1)

    _plot_fns = [(plot_acf, "ACF", lags)]
    if show_pacf:
        _plot_fns.append((plot_pacf, "PACF", pacf_lags))
    for fn, title, fn_lags in _plot_fns:

        fig, ax = plt.subplots(figsize=(12, 4))
        fn(series, lags=fn_lags, alpha=0.05, ax=ax, zero=False)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel("Korelácia")
        ax.set_xlabel("Lag (mesiace)")
        ax.axhline(0, color="black", linewidth=0.8)

        for k in range(1, lags // n + 1):
            ax.axvline(k * n, color="tomato", linestyle="--",
                       linewidth=0.9, alpha=0.7,
                       label=f"lag {k * n}" if k == 1 else None)

        ax.legend(loc="upper right", fontsize=9)
        plt.tight_layout()
        plt.show()



def plot_spectrum_histogram(dataset_path, sk_series, ap_series=None, n=12, m_hat=4, p=8):
    """
    Histogramy sigma_1/sum(sigma_i) napriec vsetkymi oknami a krajinami v datasete.
    Dve figurky: bez centrovania (center='none') a row-centrovanie (center='row').
    Zvisla ciara = priemer pre SK (a volitelne AP benchmark) seriu.

    Parametre
    ---------
    dataset_path : str          - cesta k exports.csv (napr. 30C dataset)
    sk_series    : 1-D array    - SK exportny vektor (export_vector)
    ap_series    : 1-D array alebo None - AirPassengers benchmark (volitelne)
    n            : int          - dlzka periódy (default 12)
    m_hat        : int          - velkost rolovaneho okna v periodach (default 4)
    p            : int          - pocet okien na krajinu (default 8)
    """
    import matplotlib.pyplot as plt

    dataset = load_multivariate_dataset(dataset_path)
    sk_arr  = np.asarray(sk_series, dtype=float)
    ap_arr  = np.asarray(ap_series, dtype=float) if ap_series is not None else None

    def _benchmark_mean(series, center_mode):
        m_s = len(series) // n
        vals = []
        for i in range(m_s - m_hat):
            m_i = m_s - (i + 1)
            if m_i < m_hat:
                break
            cut = series[:n * m_i]
            win = create_data_window_vector(cut, m_i, n, m_hat)
            s = process_vector_to_svd(win, center=center_mode)['s']
            if s.sum() > 0:
                vals.append(100.0 * s[0] / s.sum())
        return float(np.mean(vals)) if vals else None

    def _compute_ratios(series_dict, center_mode):
        ratios = []
        for ser in series_dict.values():
            ser = np.asarray(ser, dtype=float)
            m = len(ser) // n
            for i in range(p):
                m_i = m - (i + 1)
                if m_i < m_hat:
                    continue
                cut = ser[:n * m_i]
                win = create_data_window_vector(cut, m_i, n, m_hat)
                s = process_vector_to_svd(win, center=center_mode)['s']
                if s.sum() > 0:
                    ratios.append(100.0 * s[0] / s.sum())
        return ratios

    for center_mode, mode_label in [('none', 'bez centrovania'), ('row', 'row centrovanie')]:
        ratios   = _compute_ratios(dataset, center_mode)
        sk_mean  = _benchmark_mean(sk_arr, center_mode)
        ap_mean  = _benchmark_mean(ap_arr, center_mode) if ap_arr is not None else None

        r_mean = float(np.mean(ratios))

        fig, ax = plt.subplots(figsize=(8, 3))
        ax.hist(ratios, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
        ax.axvline(r_mean, color='#2c3e50', linewidth=1.6, linestyle='--',
                   label=f'priemer = {r_mean:.1f}%')
        if sk_mean is not None:
            ax.axvline(sk_mean, color='#e74c3c', linewidth=2.0,
                       label=f'SK = {sk_mean:.1f}%')
        if ap_mean is not None:
            ax.axvline(ap_mean, color='#2ecc71', linewidth=2.0,
                       label=f'AP = {ap_mean:.1f}%')
        ax.legend(fontsize=10)
        ax.set_xlabel(r'$\sigma_1 \,/\, \sum \sigma_i \;[\%]$', fontsize=11)
        ax.set_ylabel('Po\u010det okien', fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

